在 Megatron-LM 框架中，1F1B 调度策略的核心实现逻辑主要位于以下文件中：
- megatron/core/pipeline_parallel/schedules.py

这个文件是 Megatron-LM 流水线并行的核心调度引擎，它定义了流水线执行的所有底层逻辑。在这个文件中，你可以找到以下关键实现：
1. 1F1B 核心调度函数：
   - 通常命名为 get_forward_backward_func()，该函数会根据你配置的并行参数（如是否开启虚拟流水线并行 VPP）返回相应的调度执行器。
2. 三阶段调度循环：
   - 文件中包含了实现 1F1B 调度三个核心阶段的循环逻辑：
     - Warm-up（预热阶段）：执行初始的前向传播，将流水线填满。
     - 1F1B 稳定阶段（Steady State）：实现一次前向（Forward）紧跟一次反向（Backward）的交替执行逻辑。
     - Cool-down（冷却/收尾阶段）：执行剩余微批次的反向传播，排空流水线。
3. 通信与计算的结合：
   - 在循环内部，Megatron 还插入了点对点（P2P）的通信节点（如发送前向激活值、接收反向梯度），并结合了奇偶通信策略（Odd-even communication）来避免交叉通信带来的环状等待。

如果你需要研究 1F1B 的具体代码实现，或者想要在此基础上修改调度逻辑（例如实现 Zero Bubble 或 DualPipe 等更高级的调度算法），直接查看和修改这个 schedules.py 文件是最直接的切入点。

# get_forward_backward_func
 
- 通常命名为 get_forward_backward_func()，该函数会根据你配置的并行参数（如是否开启虚拟流水线并行 VPP）返回相应的调度执行器。


  - 根据 parallel_state（并行状态）的配置，获取合适的 forward_backward（前向-反向）函数。

  - 返回一个函数，该函数将根据全局 parallel_state 中的流水线模型并行世界大小（pipeline model parallel world size）
  - 和虚拟流水线模型并行世界大小（virtual pipeline model parallel world size），执行模型的所有前向传播和反向传播过程。

  - 注意：如果使用了序列并行（sequence parallelism），张量形状（tensor shape）中的序列长度分量
  - 将被更新为 original_sequence_length / tensor_model_parallel_world_size。

返回的函数接受以下参数：

    forward_step_func (必填): 
        一个接受数据迭代器（data iterator）和模型（model）作为参数的函数，
        返回模型的前向输出和损失函数（loss function）。
        该损失函数应接受一个 torch.Tensor，并返回一个包含损失值的 torch.Tensor 
        以及一个字典（string -> torch.Tensor）。

        第三个参数 checkpoint_activations_microbatch 指示是否应对该微批次的激活值进行检查点保存（checkpointed）。
        如果该参数为 None，则使用配置中的默认值。
        当使用 num_microbatches_with_partial_activation_checkpoints 时会用到此参数。

        示例：
```python
        def loss_func(loss_mask, output_tensor):
            losses = output_tensor.float()
            loss_mask = loss_mask.view(-1).float()
            loss = torch.sum(losses.view(-1) * loss_mask) / loss_mask.sum()

            # 对 loss 进行聚合以便记录日志。
            averaged_loss = average_losses_across_data_parallel_group([loss])

            return loss, {'lm loss': averaged_loss[0]}

        def forward_step(data_iterator, model):
            data, loss_mask = next(data_iterator)
            output = model(data)
            return output, partial(loss_func, loss_mask)


        forward_backward_func(forward_step_func=forward_step, ...)

```
    - data_iterator (必填): 
        数据迭代器，将原样传递给 forward_step_func。
        如果使用了交错式流水线并行（interleaved pipeline parallelism），则预期为一个迭代器列表。

    - model (必填): 
        实际的模型。
        如果使用了交错式流水线并行，则预期为一个模块列表。
        必须是一个（可能被包装过的）megatron.core.models.MegatronModule。

    - num_microbatches (int, 必填): 
        需要处理的微批次数量。

    - seq_length (int, 必填): 
        当前全局批次的序列长度。
        如果是双栈 Transformer（dual-stack transformer），这指的是编码器的序列长度。
        如果配置中 variable_seq_lengths 为 True，则忽略此参数。
        否则，当前全局批次中的每个微批次都必须使用此序列长度。

    - micro_batch_size (int, 必填): 
        一个微批次中包含的序列数量。

    - decoder_seq_length (int, 可选): 
        双栈 Transformer 中解码器的序列长度。对于单栈 Transformer，此参数将被忽略。

    - forward_only (可选, 默认 = False): 
        仅执行前向传播步骤。

    - collect_non_loss_data (可选, bool, 默认=False): 
        TODO（待补充）。

    - first_val_step (bool, 可选): 
        是否为验证阶段的第一步。
        供 Transformer Engine 模块使用，使其仅在验证的第一步更新 fp8 权重。

    - adjust_tensor_shapes_fn (Callable, 可选): 
        一个用于调整接收和发送张量形状的函数。
        目前仅适用于 forward_backward_pipelining_without_interleaving。
        该函数接收一个接收形状列表和一个发送形状列表，并返回调整后的相应形状列表。
        因此，它不会用于其他具有不同形状处理机制的前向-反向函数中。

    - force_all_reduce (bool, 可选): 
        如果为 True，则在本轮迭代中强制使用 all-reduce 进行梯度归约，
        而不是 reduce-scatter（在使用分布式优化器时），以确保所有数据并行（DP）rank 拥有完全归约后的梯度。
        这有助于更简单地保存 wgrad（权重梯度）
        （只需检查 DP replica 0 即可获取整个模型的完整 wgrad 集合）。

    - Args:
        pp_size (Optional[int]): 要使用的流水线模型并行大小。
        vp_size (Optional[int]): 要使用的虚拟流水线模型并行大小。
            如果 pp_size 和 vp_size 均为 None，则这两个值均回退使用 parallel_state 中的值。
            否则，将直接使用提供的值，并将 None 视为显式输入。


In [ ]:
def get_forward_backward_func(pp_size: Optional[int] = None, vp_size: Optional[int] = None):
    
    if pp_size is None and vp_size is None:
        pp_size = parallel_state.get_pipeline_model_parallel_world_size()
        vp_size = parallel_state.get_virtual_pipeline_model_parallel_world_size()

    if pp_size > 1:
        if vp_size is not None:
            forward_backward_func = forward_backward_pipelining_with_interleaving
        else:
            forward_backward_func = forward_backward_pipelining_without_interleaving
    else:
        forward_backward_func = forward_backward_no_pipelining
    return forward_backward_func


# forward_backward_pipelining_with_interleaving

- 这份代码是 Megatron-LM 框架中实现 交错式 1F1B (Interleaved 1F1B) 调度策略的核心函数 forward_backward_pipelining_with_interleaving。

- 简单来说，它通过将一个模型切分成多个“模型块”（Model Chunks），并将这些块交错地分配到不同的 GPU 上，从而实现了更高效的流水线并行训练。这能有效减少传统流水线并行中的“气泡”（即 GPU 空闲等待时间），提升训练效率。

以下是该函数工作原理的详细介绍：
## 🎯 核心目标：减少“气泡”，提升效率
- 在传统的流水线并行中，一个模型被顺序地切分到不同的 GPU 上。
  - <font color='red'>这会导致一个严重问题：当数据在 GPU 之间流动时，大部分 GPU 会处于空闲状态，形成所谓的“气泡”（Bubble）。</font>
- 交错式 1F1B 的核心思想就是“化整为零，交错排列”。
  - 它不是把一个模型分成几块，而是将一个模型虚拟地分成更多的“模型块”（例如，将一个模型分成 8 块），然后将这些块更均匀地分配到所有 GPU 上。这样，每个 GPU 都持有模型的一部分，并且可以更频繁地进行计算，从而极大地压缩了气泡，提高了 GPU 的利用率。

## 🏗️ 核心概念解析
在深入代码之前，需要先理解几个关键概念：
- <b>模型块</b> (Model Chunk): 
  - 一个完整模型被虚拟切分后的子模块。例如，一个包含 24 层的 Transformer 模型，在虚拟流水线并行度（VP Size）为 4 的情况下，会被切分成 4 个模型块，每个块包含 6 层。
- <b>虚拟微批次</b> (Virtual Microbatch): 
  - 为了调度方便，代码内部引入了一个虚拟的微批次 ID。总的虚拟微批次数量 = num_microbatches * num_model_chunks。 <font color='red'>调度器根据这个 ID 来决定当前应该处理哪个真实的微批次（microbatch_id）以及哪个模型块（model_chunk_id）。</font>
- <b>调度表</b> (Schedule Table): 一个预先计算好的查找表，它定义了每个 virtual_microbatch_id 对应哪个 microbatch_id 和 model_chunk_id。 <font color='red'>这是整个交错调度的“总指挥”。</font>

## 🔄 三大执行阶段
整个训练过程被清晰地划分为三个阶段，这与 1F1B 的基本思想一脉相承，但在交错模式下更为复杂。
### 1. 热身阶段 (Warm-up)
- 目标: 将整个流水线“灌满”，让所有 GPU 都开始工作。
- 行为: 在这个阶段，系统只执行前向传播（Forward）。代码中的 for k in range(num_warmup_microbatches): 循环负责此阶段。
- 过程:
  1. 第一个 GPU 开始计算第一个微批次的前向传播。
  2. 计算完成后，将激活值（activations）发送给下一个 GPU。
  3. 同时，它开始计算第二个微批次的前向传播。
  4. 这个过程持续进行，直到所有 GPU 都被分配了任务，并且流水线中充满了正在处理的数据。
### 2. 稳定状态 (Steady State)
- 目标: 进入高效的 1F1B 交替计算模式。
- 行为: 这是 1F1B 的核心。代码中的 for k in range(num_microbatches_remaining): 循环负责此阶段。
- 过程:
  1. 在这个阶段，每个 GPU 会交替执行 1 次前向传播 (1F) 和 1 次反向传播 (1B)。
  2. 例如，一个 GPU 完成一个微批次的前向计算后，不会立即开始下一个前向，而是等待来自下游的梯度，然后执行一个微批次的反向传播。
  3. 这种“算一个前向，紧跟着算一个反向”的模式，使得中间激活值可以被立即释放，极大地节省了显存。
  4. forward_backward_helper_wrapper 函数是执行这个 1F1B 步骤的核心，它根据调度表来决定当前是处理哪个微批次和哪个模型块。
  
### 3. 收尾阶段 (Cool-down)
- 目标: 排空流水线，完成所有剩余的计算。
- 行为: 当所有微批次的前向传播都完成后，进入此阶段。代码中的 for k in range(num_microbatches_remaining, total_num_microbatches): 循环负责此阶段。
- 过程:
  1.此时不再有新的前向任务。
  2.所有 GPU 会依次完成它们手中剩余微批次的反向传播计算。
  3.直到最后一个微批次的梯度计算完成，整个迭代结束。
  
## 💡 关键优化技术
这份代码还集成了多种高级优化技术，以追求极致的性能：
- 通信与计算重叠 (Communication Overlap):
  - 代码通过 config.overlap_p2p_comm 等配置，尝试在计算的同时进行 GPU 之间的数据发送和接收。
  - 例如，在 pp_post_forward 函数中，它会异步地发送当前计算结果并预接收下一个微批次的数据，从而隐藏通信延迟。
- 选择性激活值检查点 (Selective Activation Checkpointing):
  - 为了进一步节省显存，代码实现了 num_microbatches_with_partial_activation_checkpoints 策略。
  - 它允许对部分微批次只保存部分层的激活值，而对其他微批次则保存全部激活值。这是一种在显存和计算开销之间的权衡。
- 异步梯度/参数同步 (Asynchronous Grad/Param Sync):
  - 在 pp_pre_forward 和 pp_pre_backward 等函数中，可以看到参数同步 (param_sync_func) 和梯度同步 (grad_sync_func) 的逻辑。
  - 这些同步操作被精心安排在计算间隙，以避免阻塞主要的计算流，从而减少 GPU 的空闲时间。
  
总而言之，forward_backward_pipelining_with_interleaving 是一个高度优化的调度器，它通过交错模型块、1F1B 执行模式以及多种通信和显存优化技术，实现了在大规模分布式训练中高效利用计算资源的目标。



# 虚拟流水线并行度 (VPP)

- 虚拟流水线并行度（Virtual Pipeline Parallelism，简称 VPP）是大模型分布式训练中的一项关键技术，-- 主要用于解决传统流水线并行中 GPU 利用率低、显存占用高以及通信效率差的问题。
- <font color='red'>它通过将一个物理 GPU 上的模型层进一步逻辑切分为多个“虚拟阶段”</font>，
  - 从而在不增加硬件数量的前提下，显著提升训练效率和资源利用率。
 

## 核心原理：从“单任务”到“多任务并发”
- 在传统流水线并行（PP）中，<font color='red'>每个 GPU 只负责模型的一个连续片段</font>（例如 GPU0 负责 Layer 0-3，GPU1 负责 Layer 4-7）。
  - <font color='red'>这种模式在数据流经过流水线时，会产生大量空闲时间（即“气泡”）</font>，尤其是在微批次数量较少或模型层数较多时，GPU 利用率可能低于 50%。
- VPP 的核心思想是：<font color='red'>让一个物理 GPU 同时承担多个“虚拟阶段”的计算任务</font>。
  - 例如，当 VPP=2 时，GPU0 不再只处理 Layer 0-3，
  - 而是被划分为两个虚拟阶段：
    - 第一个阶段处理 <font color='red'>Layer 0-1</font>，
    - 第二个阶段处理 <font color='red'>Layer 2-3</font>。
  - 调度器会以<font color='blue'>“交错”</font>的方式安排这些虚拟阶段的执行顺序，<font color='red'>使得 GPU 在完成一个微批次的前向传播后，可以立即开始另一个微批次的反向传播，</font>从而大幅压缩气泡时间。
  
  - 将每个阶段的模型进一步切分为更小的"虚拟阶段"（Virtual Stages），使得单个 GPU 可以交错执行多个虚拟阶段的不同微批次，从而最大化 GPU 利用率。
  
## 关键优势：为什么需要 VPP？

### 传统流水线并行的痛点
在传统流水线并行（如 GPipe）中：
- 模型按层（Layer）划分为多个阶段（Stage），每个阶段分配到不同的 GPU
- 一个批次（Batch）被细分为多个微批次（Micro-batch）
- 问题：每个 GPU 同一时间只能处理一个微批次，导致大量空闲时间（气泡）

### 传统 PP 时间线：

In [ ]:

GPU 0: [F0][F1][F2]      [B0][B1][B2]      ← 前向/反向传播
GPU 1:      [F0][F1][F2]      [B0][B1][B2]  ← 大量空闲等待
GPU 2:           [F0][F1][F2]      [B0][B1]  ← 气泡严重

## VPP 的工作原理
<font color='blue'>每个GPU负责的层变少（由连续的4个层，变为连续的2个层），所以等待的时间就变短了！</font>
###  虚拟阶段切分
假设模型有 16 层，使用 4 个 GPU 进行流水线并行：
- <font color='red'>每个 GPU 现在负责 2 个不连续的虚拟阶段（VPP=2）</font>

| 方案              | 每层 GPU 负责的层                                                                   |
| --------------- | ----------------------------------------------------------------------------- |
| 传统 PP           | GPU 0: 1-4, GPU 1: 5-8, GPU 2: 9-12, GPU 3: 13-16                             |
| **VPP (VPP=2)** | GPU 0: 1-2 + <font color='red'>9-10</font>, GPU 1: 3-4 + <font color='red'>11-12</font>, GPU 2: 5-6 + <font color='red'>13-14</font>, GPU 3: 7-8 + <font color='red'>15-16</font> |


###  交错执行时间线
- F0-1: 第 0 个微批次在虚拟阶段 1 的前向传播
- F0-2: 第 0 个微批次在虚拟阶段 2 的前向传播
- F1-1: 第 1 个微批次在虚拟阶段 1 的前向传播
- F1-2: 第 1 个微批次在虚拟阶段 2 的前向传播
- 同一 GPU 交错处理不同虚拟阶段，几乎消除空闲时间

In [ ]:
VPP=2 时间线（以 GPU 0 为例）：
时间:  t1    t2    t3    t4    t5    t6    t7    t8
GPU0: [F0-1][F1-1][F0-2][F1-2][B0-2][B1-2][B0-1][B1-1]
       ↑前向阶段1   ↑前向阶段2   ↑反向阶段2   ↑反向阶段1

### VPP 的关键参数

| 参数                                | 含义              | 影响              |
| --------------------------------- | --------------- | --------------- |
| `vpp_size` / `num_virtual_stages` | 每个 GPU 上的虚拟阶段数  | 越大，气泡越小，但内存开销增加 |
| `micro_batch_size`                | 每个微批次的样本数       | 影响流水线深度         |
| `pipeline_model_parallel_size`    | 物理流水线并行度（GPU 数） | 与 VPP 共同决定总并行效率 |


### VPP 大小 vs 气泡比例

In [ ]:
气泡比例 ≈ (VPP_size - 1) / (VPP_size × num_micro_batches)

当 VPP=1（传统 PP）: 气泡最大
当 VPP=2: 气泡减少约 50%
当 VPP=4: 气泡进一步减少，但收益递减



## 实际应用：VPP 在大模型训练中的表现
在实际工程中，VPP 已被广泛应用于千亿参数模型的训练中。例如：
- 在使用 64 张 A100 GPU 训练 Qwen3-MoE-7B 模型时，<font color='red'>启用 VPP 后，训练速度提升了 8-10 倍，GPU 利用率从 60% 提升至接近 90%。</font>
- 在处理 32K 长度的长序列时，VPP 与 FlashAttention 结合，有效缓解了 KV 缓存压力，避免了 OOM（显存溢出）问题。
- 在多模态模型（如 InternVL3）训练中，<font color='green'>VPP 与异构计算优化结合，解决了负载不均衡问题，确保了训练的稳定性。</font>


## 配置建议：如何设置 VPP？
在实际使用中，VPP 的配置通常通过框架的参数进行控制。例如，在 Megatron-LM 或 ms-swift 框架中，可以通过以下参数设置：
- --virtual-pipeline-model-parallel-size：指定 VPP 的大小（通常为 2 或 4）。
- --pipeline-model-parallel-size：指定物理流水线的并行度。
- --num-microbatches：指定微批次的数量，需与 VPP 配合调整以达到最佳效果。

需要注意的是，VPP 并非越大越好。过大的 VPP 会增加调度复杂度和通信开销，反而可能降低性能。通常建议根据模型规模、GPU 数量和微批次大小进行实验性调优。

## Megatron-LM 中的 VPP
- NVIDIA 的 Megatron-LM 框架是 VPP 的主要实现者：

In [ ]:
# Megatron-LM 配置示例
--pipeline-model-parallel-size 4    # 4 个物理阶段
--num-virtual-stages 2             # VPP=2，每个 GPU 2 个虚拟阶段
--micro-batch-size 1               # 微批次大小

## 典型配置建议
| 模型规模    | GPU 数 | 推荐 VPP |
| ------- | ----- | ------ |
| 7B-13B  | 8-16  | 1-2    |
| 30B-70B | 32-64 | 2-4    |
| 100B+   | 128+  | 4-8    |


# 举例
为了让你更直观地理解 交错式 1F1B（Interleaved 1F1B） 调度，我们构建一个具体的场景来进行演示。
假设我们要训练一个大模型，配置如下：
- GPU 数量：2 张（GPU 0, GPU 1）。
- 模型总层数：8 层（Layer 0 ~ Layer 7）。
- 虚拟流水线并行度 (VPP)：2（即 num_model_chunks = 2）。
  - <font color='red'>这意味着每个 GPU 不持有连续的 4 层，而是持有“首尾各 2 层”。</font>
  - 每次负责处理的层变为2（原来是4层）
- 微批次数量 (Microbatches)：4（即 num_microbatches = 4）。

## 🧩 第一步：切分模型块与分配
根据 VPP=2，我们将 8 层模型切分为 4 个 模型块（Model Chunks），每个块包含 2 层。分配方式采用“交错”策略：
| 模型块 ID | 包含层数 | 分配给 | 备注 |
| :--- | :--- | :--- | :--- |
| Chunk 0 | Layer 0-1 | GPU 0 | 负责处理数据的输入和中间传递 |
| Chunk 1 | Layer 2-3 | GPU 1 | 负责处理数据的中间计算 |
| Chunk 2 | Layer 4-5 | GPU 0 | 再次回到 GPU 0，处理深层特征 |
| Chunk 3 | Layer 6-7 | GPU 1 | 最后由 GPU 1 输出结果并计算 Loss |

关键点： GPU 0 现在同时拥有 Chunk 0 和 Chunk 2；GPU 1 同时拥有 Chunk 1 和 Chunk 3。这就是“交错”的物理基础。

## 🔢 第二步：理解虚拟微批次
为了让调度器统一指挥，我们不直接说“第几个数据”，而是引入 虚拟微批次 ID。
- 真实微批次数量：4 个（记为 M0, M1, M2, M3）。
- 模型块数量：2 个/卡。
- 虚拟微批次总数：4×2=8 个。
调度表会将这 8 个虚拟步骤映射回真实的执行任务。例如，当调度器发出指令“执行虚拟批次 0”时，实际上是在告诉 GPU 0：“请拿 真实批次 M0 的数据，去跑你的 第一个块 Chunk 0”。

## ⏱️ 第三步：时间轴上的执行流（核心）
在交错模式下，GPU 不再像传统模式那样“干等”。以下是简化的执行顺序（省略了部分通信细节，聚焦计算逻辑）：
- 阶段 1：热身期
目标是将流水线填满，让所有 GPU 都有活干。
  - T1: GPU 0 执行 M0 + Chunk 0。
  - T2:
    - GPU 0 执行 M0 + Chunk 2（注意：这里 M0 的数据已经流过 GPU 1 传回来了，或者说是前向传播的后续部分）。
    - GPU 1 执行 M0 + Chunk 1（接收 T1 的输出）。
  - T3:
    - GPU 0 执行 M1 + Chunk 0（开始处理新数据）。
    - GPU 1 执行 M0 + Chunk 3（M0 完成前向，计算 Loss）。
    - GPU 1 执行 M1 + Chunk 1（紧接着处理 M1 的第一部分）。
现象观察： 在传统模式下，T3 时刻 GPU 0 可能还在等 GPU 1 的消息，但在交错模式下，因为 GPU 0 手里还有 Chunk 2 的任务没做完（或者刚做完准备接新活），它能更紧凑地安排任务。
- 阶段 2：稳定期
这是 1F1B 的精髓：做一个前向，紧接着做一个反向。
  - T4:
    - GPU 0: 前向 M2 + Chunk 0，紧接着反向 M0 + Chunk 2。
    - GPU 1: 前向 M2 + Chunk 1，紧接着反向 M0 + Chunk 3。
此时，流水线完全饱和。每个 GPU 都在全速运转，没有空闲气泡。
- 阶段 3：收尾期
当前向全部发完后，开始清理剩余的反向传播。
   - TN: GPU 1 执行 M3 + Chunk 3 的反向（最后一个任务）。
   - TN+1: GPU 0 执行 M3 + Chunk 2 的反向。

# 构建流水线并行（PP）所需的通信基础设施

In [ ]:
┌─────────────────────────────────────────────────────────┐
│                    初始化流程                             │
├─────────────────────────────────────────────────────────┤
│  首次调用 (VPP 虚拟阶段 0)                                │
│    → 创建 P2PCommunicator（相邻阶段间点对点通信）         │
│    → 收集所有并行通信组 (TP/CP/PP/DP/Embd)               │
│    → 封装为 ProcessGroupCollection                       │
├─────────────────────────────────────────────────────────┤
│  复用调用 (VPP 虚拟阶段 1,2,3...)                         │
│    → 直接取用已有 pg_collection                          │
│    → 避免重复创建 MPI/NCCL 通信组（昂贵操作）              │
└─────────────────────────────────────────────────────────┘

# 与 VPP 调度时序的关系

In [ ]:
校验通过后，进入 VPP 核心调度：

时间 →
GPU 0: [F0_chunk0][F1_chunk0][F0_chunk1][F1_chunk1][B0_chunk1][B1_chunk1][B0_chunk0][B1_chunk0]
             ↑        ↑        ↑        ↑        ↑        ↑        ↑        ↑
          微批次0   微批次1   微批次0   微批次1   微批次0   微批次1   微批次0   微批次1
          阶段0前向  阶段0前向  阶段1前向  阶段1前向  阶段1反向  阶段1反向  阶段0反向  阶段0反向

每个 [Fx_chunky] 需要：
  1. 从 data_iterator[y] 取数据
  2. 通过 model[y] 前向计算
  3. 通过 p2p_communicator 发送/接收激活值

# 什么是梯度最终化（Finalize Model Grads）？
- 将分散的梯度聚合起来

In [ ]:
标准 PyTorch 梯度流：
  loss.backward() → 梯度 Accumulate 到 .grad → 直接可用

Megatron 分布式场景：
  loss.backward() → 梯度分散在各 PP/TP 阶段 → 需要**最终化** → 统一可用状态

## 为什么需要最终化？

| 并行策略           | 梯度问题                      | 最终化操作                     |
| -------------- | ------------------------- | ------------------------- |
| **张量并行 (TP)**  | 同一层被切分到多 GPU，梯度是**部分和**   | All-Reduce 聚合完整梯度         |
| **流水线并行 (PP)** | 嵌入层在首阶段，输出层在末阶段，梯度**跨阶段** | 特殊同步逻辑                    |
| **序列并行 (SP)**  | 序列维度切分，梯度需**跨序列维度**聚合     | All-Gather/Reduce-Scatter |


# 嵌入层激活缓冲清理
## 为什么需要清理嵌入层激活缓冲？
- <font color='red'>嵌入层的输出张量（embeddings）：作为中间激活值，需要保存到下游层反向传播完成</font>
- 不是嵌入层需要保存"自己的状态"，而是嵌入层输出的 embeddings 张量作为中间激活值，在 VPP 交错调度下会累积多个微批次的副本，需要及时清理释放内存。

In [ ]:
传统 PP（无 VPP）：
  [F0] → embeddings_0 → Block0 → ... → 立即 [B0] → 释放 embeddings_0
  内存峰值：1 个 embeddings 张量

VPP=2（交错）：
  [F0_chunk0] → embeddings_0 → Block0_chunk0
  [F1_chunk0] → embeddings_1 → Block0_chunk0
  [F0_chunk1] → embeddings_2 → Block0_chunk1  ← 注意：这是另一个虚拟阶段
  [F1_chunk1] → embeddings_3 → Block0_chunk1
  
  然后才开始反向传播...
  
  内存峰值：4 个 embeddings 张量同时存在！

clear_embedding_activation_buffer 的作用：
  → 在合适的时机释放这些累积的 embeddings 张量
  → 防止 OOM

In [ ]:
VPP 交错调度中的嵌入层问题：

时间 →
Stage 0 (GPU 0): [F0_chunk0][F1_chunk0] ... [F0_chunk1][F1_chunk1]
                  ↑ 微批次0使用嵌入层    ↑ 微批次1使用嵌入层
                  
嵌入层特点：
  - 通常只在 PP 首阶段（Stage 0）
  - 输入是 token IDs，输出是 embeddings
  - 需要保存激活值用于反向传播梯度计算

VPP 问题：
  - 多个微批次交错执行，嵌入层激活值**累积在缓冲中**
  - 如果不清理，内存爆炸（OOM）
  - 如果清理过早，反向传播时找不到激活值

### 核心区别：嵌入层 vs 嵌入层输出

In [ ]:
输入: token_ids [batch, seq_len]          ← 整数索引
      ↓
嵌入层: nn.Embedding(vocab_size, hidden_dim)  ← 查表操作
      ↓
输出: embeddings [batch, seq_len, hidden_dim]  ← 这个输出需要保存！

问题：保存的是"输出张量"还是"嵌入层的内部状态"？
答案：保存的是"输出张量"（用于反向传播链式法则）

### 为什么嵌入层输出需要保存？
1. 反向传播链式法则

In [ ]:
前向传播链：
token_ids → [Embedding] → embeddings → [Transformer Block 0] → ... → [LM Head] → logits → loss

反向传播链：
loss.grad ← logits.grad ← ... ← block0_output.grad ← **embeddings.grad** ← ?

关键问题：embeddings.grad 怎么传播到 token_ids？
答案：不需要传播到 token_ids！

| 层类型           | 输入是否需要梯度 | 原因                 |
| ------------- | -------- | ------------------ |
| Linear        | 是        | 输入是上一层的输出，需要继续反向传播 |
| Attention     | 是        | 同上                 |
| **Embedding** | **否**    | 输入是整数索引，不可微        |


In [ ]:
Embedding 的反向传播：
  - 输入 token_ids 是整数 → 无梯度
  - 输出 embeddings 有梯度 → 但只用于更新 Embedding 权重矩阵
  - 梯度流向：embeddings.grad → 聚合到对应词向量的权重梯度
  
实际上：Embedding 层不需要保存输出激活值用于"反向传播给前一层的梯度"
        因为前一层的输入是整数，不需要梯度

### 那为什么代码里要清理？
- 真正的原因：VPP 的交错调度导致激活值累积

In [ ]:
VPP=2 场景（一个 GPU 两个虚拟阶段）：

时间 →
GPU 0 (虚拟阶段 0 + 虚拟阶段 2):
  [F0_chunk0] 使用嵌入层 → 输出 embeddings_0 → 传给 Transformer
  [F1_chunk0] 使用嵌入层 → 输出 embeddings_1 → 传给 Transformer
  [F0_chunk2] 不使用嵌入层（中间层）
  [F1_chunk2] 不使用嵌入层
  [B1_chunk2] 反向传播
  [B0_chunk2] 反向传播
  [B1_chunk0] 反向传播 ← 需要 embeddings_1 的激活值？
  [B0_chunk0] 反向传播 ← 需要 embeddings_0 的激活值？

问题：embeddings_0 和 embeddings_1 什么时候可以释放？

### 关键：嵌入层输出是中间激活值，不是嵌入层内部状态

```python
# 这里的"激活值"指的是：
embeddings = embedding_layer(token_ids)  # 这个 embeddings 张量

# 在 Transformer Block 0 的反向传播中：
block0_output.grad → 需要 block0_input.grad
block0_input = embeddings + pos_emb  # 需要 embeddings 的值来计算梯度

# 所以 embeddings 张量需要保存到 block0 反向完成
```

## clear_embedding_activation_buffer 的作用：

In [ ]:
时机：所有微批次前向完成，反向传播开始前/后

操作：
  1. 识别当前阶段的嵌入层模块
  2. 检查是否已用完所有微批次的反向传播
  3. 安全释放嵌入层激活缓冲 → 回收内存

vp is ignored: 无论 VPP 虚拟阶段数多少，嵌入层只在物理首阶段存在一次

## is_pp_last_stage 的用途

In [ ]:
PP 阶段分布：
  Stage 0 (首阶段): 嵌入层 + 初始 Transformer 层
  Stage 1: 中间层
  ...
  Stage N-1 (末阶段): 最终层 + LM Head/输出层

为什么关注末阶段？
  - 某些嵌入层逻辑（如 tied embedding）需要与末阶段同步
  - 判断是否需要参与嵌入权重的梯度聚合

# 完整流程图

In [ ]:
VPP=2 训练步骤：

1. 初始化（本代码）
   no_sync_func = multi_no_sync()  ← 合并两个 chunk 的 no_sync
   grad_sync_func = [sync_0, sync_1]
   param_sync_func = [param_sync_0, param_sync_1]

2. 进入 no_sync 模式
   with no_sync_func():  ← 同时禁用 chunk0 和 chunk1 的梯度同步
   
   3. 交错前向传播
      [F0_c0] → [F1_c0] → [F0_c1] → [F1_c1]
      
   4. 交错反向传播
      [B1_c1] → [B0_c1] → [B1_c0] → [B0_c0]
      ↑ 梯度累加到本地，不触发 All-Reduce

5. 退出 no_sync 模式
   ← 自动触发梯度同步
   
6. 手动梯度同步（如有需要）
   for sync in grad_sync_func:
       sync()  ← 确保所有 chunk 梯度已同步
   
7. 优化器 step
   optimizer.step()
   
8. 参数同步（ZeRO-3 场景）
   for param_sync in param_sync_func:
       param_sync()  ← 同步更新后的参数

# SP 与 TP 的区别：

|    | 切分维度                | 切分对象  | 通信操作                      |
| -- | ------------------- | ----- | ------------------------- |
| TP | hidden\_size (最后一维) | 权重/激活 | All-Reduce                |
| SP | seq\_length (第一维)   | 激活值   | All-Gather/Reduce-Scatter |
| CP | seq\_length (第一维)   | 注意力计算 | All-Gather                |


In [ ]:
import contextlib
from megatron.core import parallel_state
from megatron.core.pipeline_parallel.fine_grained_activation_offload import (
    FineGrainedActivationOffloadingInterface as off_interface,
)

def forward_backward_pipelining_with_interleaving(
    *,
    forward_step_func,
    data_iterator: Union[Iterator, List[Iterator]],
    model: Union[torch.nn.Module, List[torch.nn.Module]],
    num_microbatches: int,
    seq_length: int,
    micro_batch_size: int,
    decoder_seq_length: Optional[int] = None,
    forward_only: bool = False,
    collect_non_loss_data: bool = False,
    first_val_step: Optional[bool] = None,
    adjust_tensor_shapes_fn: Optional[Callable] = None,  # unused
    p2p_communicator: Optional[P2PCommunicator] = None,
    pg_collection: Optional[ProcessGroupCollection] = None,
    force_all_reduce: Optional[bool] = False,
):
    """Run interleaved 1F1B schedule (model split into model chunks), with
    communication between pipeline stages as needed.

    Returns dictionary with losses if the last stage, empty dict otherwise."""

    # Convention used in this function:
    # num_microbatches for number of microbatches per pipeline stage;
    # num_model_chunks for virtual pipeline size;
    # then total_num_microbatches = num_microbatches * num_model_chunks.
    # Their corresponding index variables are
    # microbatch_id in [0, num_microbatches)
    # model_chunk_id in [0, num_model_chunks)
    # virtual_microbatch_id in [0, total_num_microbatches)
    
    # 本函数中使用的命名约定如下：
    # num_microbatches：每个流水线阶段（pipeline stage）所处理的微批次数量；
    # num_model_chunks：虚拟流水线并行大小（virtual pipeline size），即模型块的数量；
    # 因此，总的微批次数量 total_num_microbatches = num_microbatches * num_model_chunks。
    # 
    # 它们对应的索引变量分别为：
    # microbatch_id：真实微批次 ID，取值范围为 [0, num_microbatches)
    # model_chunk_id：模型块 ID，取值范围为 [0, num_model_chunks)
    # virtual_microbatch_id：虚拟微批次 ID，取值范围为 [0, total_num_microbatches)

    config = get_model_config(model[0])
    
    '''
    两个通信基础设施都未创建（首次调用）
    '''
    if p2p_communicator is None and pg_collection is None:
        '''
        创建 P2P 通信器:点对点通信器，处理相邻流水线阶段间的激活值和梯度传输
          
        pp_group	流水线并行的进程组（包含所有参与 PP 的 GPU）
        '''
        p2p_communicator = P2PCommunicator(
            pp_group=parallel_state.get_pipeline_model_parallel_group(), config=config
        )
        
        '''
        收集所有并行通信组
        '''
        tp_group = parallel_state.get_tensor_model_parallel_group()
        cp_group = parallel_state.get_context_parallel_group()
        cp_size = cp_group.size # CP 并行度
        embd_group = parallel_state.get_embedding_group(check_initialized=False) # 嵌入层通信组
        pp_group = parallel_state.get_pipeline_model_parallel_group()
        '''
        check_initialized=False：允许通信组未初始化时不报错（兼容某些配置）
        '''
        pos_emb_group = parallel_state.get_position_embedding_group(check_initialized=False) # 位置编码通信组

        '''
         封装为集合对象
         
         ProcessGroupCollection 是一个聚合对象，将所有通信组统一管理：
        '''
        pg_collection = ProcessGroupCollection()
        pg_collection.tp = tp_group
        pg_collection.cp = cp_group
        pg_collection.embd = embd_group
        pg_collection.pos_embd = pos_emb_group
        pg_collection.pp = pp_group
        '''
        dp_cp 的特殊性：
        - 当使用 CP 时，DP 需要跨 CP 组进行
        - 例如：8 GPU，CP=2，DP=4 → dp_cp 包含 4 个逻辑副本
        '''
        pg_collection.dp_cp = parallel_state.get_data_parallel_group(
            with_context_parallel=True, partial_data_parallel=False
        )

    elif p2p_communicator is not None and pg_collection is not None:
        '''
        通信基础设施已存在（非首次调用，如虚拟流水线 VPP 的其他虚拟阶段）
        '''
        model_type = get_model_type(model[0]) # 检查模型类型（GPT、T5、BERT 等）
        '''
        从已有 pg_collection 提取，避免重复创建
        
        为什么需要复用？
        
        VPP=2 时，一个物理 GPU 有两个虚拟阶段：
        虚拟阶段 A → 首次调用，创建通信器
        虚拟阶段 B → 复用已有通信器（同 GPU，同进程组）
        '''
        assert hasattr(p2p_communicator, 'config'), "p2p_communicator must have a config"
        assert hasattr(pg_collection, 'tp'), "pg_collection must have tp"
        assert hasattr(pg_collection, 'cp'), "pg_collection must have cp"
        tp_group = pg_collection.tp
        cp_group = pg_collection.cp
        cp_size = cp_group.size()
    else:
        '''
        确保要么全创建，要么全复用
        '''
        raise ValueError(
            "Invalid combination of p2p_communicator, pg_collection"
            " provide none or provide all the process groups"
        )

    '''
    模型必须是列表（虚拟阶段列表）
    
    列表，每个元素是一个虚拟阶段
    VPP (Interleaved): model = [GPTModel_chunk0, GPTModel_chunk1, ...]
    
    为什么必须是列表？
    
    VPP=2 时，一个物理 GPU 负责两个不连续的层段：
    model[0] → 虚拟阶段 0 (Layers 1-2)
    model[1] → 虚拟阶段 1 (Layers 5-6)
    
    每个虚拟阶段是独立的 nn.Module 实例，需要分别前向/反向传播。
    '''
    assert isinstance(model, list), "interleaved pipeline parallelism expected model chunking"
    '''
    每个元素必须是有效的 PyTorch 模块
    防御性校验：确保列表中的每个"chunk"都是真正的神经网络模块。
    
    
    model = [
        GPTModel(config, layers=1-2),   ← 必须是 nn.Module
        GPTModel(config, layers=5-6),   ← 必须是 nn.Module
    ]
    '''
    assert all(isinstance(chunk, torch.nn.Module) for chunk in model), "invalid model chunking"
    '''
    数据迭代器必须是列表（与模型一一对应）
    '''
    assert isinstance(
        data_iterator, list
    ), "interleaved pipeline parallelism expected each model chunk to have a data iterator"
    
    '''
    禁用张量形状调整函数:
    - 传统 PP:可用（手动调整通信张量形状）
    - VPP: 禁止
    
    为什么 VPP 不支持？
    
    VPP 的通信复杂度远高于传统 PP：
     - 相邻虚拟阶段间通信
     - 同一 GPU 内不同虚拟阶段间状态切换
     - 微批次交错调度

     手动形状调整容易破坏 VPP 的精密调度时序。
    '''
    assert (
        adjust_tensor_shapes_fn is None
    ), "adjust_tensor_shapes_fn is not supported for interleaved pipeline parallelism"

    '''
    通信优化选项互斥
    
    overlap_p2p_comm	重叠 P2P 通信与计算	异步通信，计算和通信并行
    batch_p2p_comm	批量 P2P 通信	合并多个小通信为一次大通信

    为什么不能同时开启？
    
    场景：GPU 0 需要发送激活值给 GPU 1

    overlap_p2p_comm:
      计算 micro-batch 2  ──并行──→  发送 micro-batch 1 的激活值
      [计算和通信同时进行]

    batch_p2p_comm:
      收集 micro-batch 1,2,3 的激活值 → 一次性发送
      [延迟发送，合并批量]

    冲突点：
      - overlap 要求"立即发送"以隐藏延迟
      - batch 要求"延迟发送"以合并批量
      → 调度策略矛盾，同时开启会导致死锁或错误
    '''
    if config.overlap_p2p_comm and config.batch_p2p_comm:
        raise ValueError("Can not use both overlap_p2p_comm and batch_p2p_comm")

    # Needed only when gradients are finalized in M-Core
    '''
    VPP 训练步骤完整流程    
    
    1. 初始化校验（上段代码）                                    
    2. 前向传播交错调度（多个虚拟阶段）                          
    3. 反向传播交错调度                                          
    4. 梯度最终化 ←── 本代码所在位置                              
    5. 性能计时统计
    
    '''
    '''
    梯度最终化条件判断:
    
    config.finalize_model_grads_func is not None: M-Core（Megatron Core）需要自定义梯度最终化逻辑
    not forward_only: 不是仅前向推理（即训练模式，需要计算梯度）
    
    
    '''
    if config.finalize_model_grads_func is not None and not forward_only:
        '''
        嵌入层激活缓冲清理:
        
        config	模型配置，包含嵌入层类型、梯度最终化函数等
        model	VPP 的模型列表 [chunk0, chunk1, ...]
        is_pp_last_stage(...)	判断当前 GPU 是否是流水线最后一个阶段
        '''
        # vp is ignored for clear_embedding_activation_buffer
        embedding_module = clear_embedding_activation_buffer(
            config, model, is_pp_last_stage(p2p_communicator.pp_group)
        )

    if config.timers is not None:
        config.timers('forward-backward', log_level=1).start(barrier=config.barrier_with_L1_time)

    '''
    核心目的是管理分布式训练中的梯度聚合时机，避免 VPP 交错调度导致的梯度同步冲突:
    
    这段代码的核心是：将单 chunk 的梯度/参数同步控制，扩展为 VPP 多 chunk 的批量控制，
    确保交错调度下"先累加再同步"的正确性，避免梯度不完整或同步冲突。
    
    核心概念：为什么需要禁用异步梯度归约？
    
    VPP 交错调度下，一个 GPU 有多个虚拟阶段：
    
    如果每个 micro-batch 反向完就 All-Reduce：
    - micro-batch 0 chunk0 的梯度还没算完
    - micro-batch 1 chunk0 的梯度就开始 All-Reduce 了
    → 梯度不完整！错误！
    
    
    解决方案：no_sync 模式：
    
    
    no_sync 模式（PyTorch DDP 提供）：
      with ddp_model.no_sync():
          loss.backward()  ← 梯度只累加到本地，不触发 All-Reduce
  
    # 所有 micro-batch 反向完成后
    ddp_model.grad_sync()  ← 统一触发一次 All-Reduce
    
    '''
    # Disable async grad reductions
    '''获取 no_sync 函数'''
    no_sync_func = config.no_sync_func
    if isinstance(no_sync_func, list):
        '''VPP 多虚拟阶段	列表，每个 chunk 一个'''
        '''
        为什么 VPP 需要列表？
        
        VPP=2 时，一个物理 GPU 有两个虚拟阶段：
          model = [chunk0, chunk1]
  
        每个 chunk 可能是独立的 DDP 实例：
          chunk0 → ddp_model_0 → ddp_model_0.no_sync()
          chunk1 → ddp_model_1 → ddp_model_1.no_sync()

        需要同时进入两个 no_sync 上下文！
        '''
        def multi_no_sync():
            '''
            使用 ExitStack（动态、可扩展）：
            
            不使用 ExitStack（嵌套写法，丑陋且不可扩展）：
              with chunk0.no_sync():
                  with chunk1.no_sync():
                      with chunk2.no_sync():
                          # ... 反向传播
                          pass
            所有 no_sync 同时生效
            stack 关闭时，所有上下文自动退出
            '''
            stack = contextlib.ExitStack()
            for model_chunk_no_sync_func in config.no_sync_func:
                # 把每个chunk都设置为禁止同步
                # 当所有chunk都完成反向传播时，退出上下文管理模式，立即触发同步
                stack.enter_context(model_chunk_no_sync_func())
            return stack

        no_sync_func = multi_no_sync
        
    '''
    没有提供 no_sync 功能：
     使用场景：非 DDP 训练（如纯 TP/PP，没有数据并行）
    
    '''
    if no_sync_func is None:
        '''什么都不做的上下文管理器'''
        no_sync_func = contextlib.nullcontext
        
    '''
    延迟初始化：在实际的训练循环中再创建：
    '''
    no_sync_context = None

    '''
    梯度同步函数列表化
    '''
    if config.grad_sync_func is not None and not isinstance(config.grad_sync_func, list):
        
        '''
        grad_sync_func	手动触发梯度 All-Reduce 的函数 (触发时机	反向传播完成后)
        model	VPP 的模型列表 [chunk0, chunk1, ...]

        为什么需要列表化？
        
        不同 chunk 不同同步策略
        '''
        config.grad_sync_func = [config.grad_sync_func for _ in model] 

    if config.param_sync_func is not None and not isinstance(config.param_sync_func, list):
        
        '''
        为什么参数也需要同步？
        
        ZeRO-3 场景：
          每个 GPU 只保存部分参数
          前向传播前：All-Gather 聚合完整参数
          反向传播后：Reduce-Scatter 分发梯度
          优化器 step 后：参数更新，需要同步新参数值

        VPP 下多个 chunk 可能有独立的参数分片 → 需要独立的 sync 函数
        
        '''
        config.param_sync_func = [config.param_sync_func for _ in model]

    # Disable config.grad_sync_func and config.param_sync_func if only running forward passes.
    # They will be re-enabled at the end of this function.
    '''
    核心目的是在推理模式（forward_only）下完全禁用梯度同步，在训练模式下按需开关同步
    
    推理模式 (forward_only=True)→ 不需要梯度 → 不需要同步 → 彻底禁用所有同步机制
    
    训练模式 (forward_only=False)                               
     → 需要梯度 → 但 VPP 交错调度需要"先累加再同步"             
     → disable_grad_sync() 进入 no_sync 模式                   
     → 所有 micro-batch 反向完成后 → enable_grad_sync() 恢复  
    
    '''
    grad_sync_func, param_sync_func = None, None
    '''推理模式：'''
    if forward_only:
        '''
        为什么需要"暂存并清空"？
        
        推理时(不需要)	推理后恢复
        
        原因：防止下游代码误触发同步
        
        暂存到局部变量：后续如果确实需要（虽然推理时不需要），局部变量保留引用
        '''
        grad_sync_func, param_sync_func = config.grad_sync_func, config.param_sync_func
        config.grad_sync_func, config.param_sync_func = None, None

    def disable_grad_sync():
        '''
        进入 no_sync 模式
        
        nonlocal	外层函数（闭包）的局部变量，非全局
        
        闭包设计：disable/enable 作为内部函数，共享外层的 no_sync_context 状态。
        '''
        """Disable asynchronous grad reductions"""
        nonlocal no_sync_context
        if no_sync_context is None: # 防御性编程，防止重复进入
            no_sync_context = no_sync_func()
            no_sync_context.__enter__()

    def enable_grad_sync():
        '''
        恢复梯度同步
        '''
        """Enable asynchronous grad reductions"""
        nonlocal no_sync_context
        if no_sync_context is not None:
            '''
            __exit__(None, None, None) 的含义
            
            __exit__(self, exc_type, exc_val, exc_tb):
                     exc_type: 异常类型（如 ZeroDivisionError）
                     exc_val: 异常对象
                     exc_tb: 异常追踪栈
            
            手动调用 __exit__ 表示"正常退出"，无异常
            '''
            no_sync_context.__exit__(None, None, None)
            no_sync_context = None

    '''
    立即禁用梯度同步
      调用时机：在 VPP 调度函数开头，所有 micro-batch 处理之前。

    '''
    disable_grad_sync()

    # Model chunk IDs with synchronized grads
    '''
    负责建立多虚拟阶段的状态跟踪体系和微批次分组约束校验
    
    ┌─────────────────────────────────────────────────────────────┐
    │              VPP 调度器状态初始化                             │
    ├─────────────────────────────────────────────────────────────┤
    │  1. 梯度同步跟踪（synchronized_model_chunks）                │
    │  2. 张量缓存池（input/output tensors）                       │
    │  3. Token 计数器（total_num_tokens）                         │
    │  4. 前向数据存储（forward_data_store）                       │
    │  5. 梯度张量缓存（output_tensor_grads）                      │
    │  6. PP 拓扑信息（size/rank）                                 │
    │  7. 微批次分组约束校验                                       │
    └─────────────────────────────────────────────────────────────┘
    
    
    '''
    
    '''
    用途：记录哪些虚拟阶段（model chunk）的梯度已经完成同步
    set 自动去重
    
    VPP 交错调度中，同一 chunk 可能多次触发同步尝试，set 自动去重避免重复记录。
    '''
    synchronized_model_chunks = set()

    input_tensors = [[] for _ in range(len(model))]
    output_tensors = [[] for _ in range(len(model))]
    
    '''
    torch.zeros([])	零维张量（标量），用于跨 GPU 聚合
    dtype=torch.int	整数计数
    device="cuda"	放在 GPU 上，方便 All-Reduce
    
    用途：
        - 统计实际处理的 token 数量（考虑 padding 后的有效 token）
        - 用于损失函数聚合（loss scaling）和吞吐量计算
        
        # 后续代码中：
          total_num_tokens += num_tokens_in_micro_batch
          
        # 最后 All-Reduce 得到全局总 token 数
          loss = total_loss / total_num_tokens  # 全局平均损失
    '''
    total_num_tokens = torch.zeros([], dtype=torch.int, device="cuda")

    '''
    前向数据存储:
    用途：存储前向传播中需要跨阶段传递或最终返回的数据
    
    存储内容可能包括：
    - 中间激活值（用于无 checkpointing 的反向传播）
    - 注意力 mask
    - 位置编码
    - 损失值（最后一个 PP stage）
    '''
    forward_data_store = []
    
    '''
    梯度张量缓存
    '''
    output_tensor_grads = None
    if not forward_only:
        output_tensor_grads = [[] for _ in range(len(model))]
    else:
        '''推理模式不需要梯度'''
        output_tensor_grads = None

    '''
    PP 拓扑信息
    
    pipeline_parallel_rank	当前 GPU 在 PP 中的位置	0, 1, 2, 3
    
    
    用途：
     判断当前 GPU 是否是首阶段（rank=0）或末阶段（rank=PP-1）
     决定是否需要接收输入或发送输出
     计算 VPP 调度中的通信对等体
    '''
    pipeline_parallel_size = p2p_communicator.pp_group.size()
    pipeline_parallel_rank = p2p_communicator.pp_group.rank()

    ''' 
    微批次分组约束校验
    
    microbatch_group_size_per_vp_stage	每个虚拟阶段内，连续处理的微批次数量
    num_microbatches	总微批次数量（总 batch / micro_batch_size）
    pipeline_parallel_size	PP 并行度
    
    
    必须满足: PP ≤ microbatch_group_size_per_vp_stage ≤ M

    即: pipeline_parallel_size ≤ G ≤ num_microbatches
    
    为什么有这个约束？
    下限 G ≥ PP：保证流水线填满
    上限 G ≤ M：不能超过总微批次
    '''
    if (
        config.microbatch_group_size_per_vp_stage > num_microbatches
        or config.microbatch_group_size_per_vp_stage < pipeline_parallel_size
    ):
        msg = (
            'The number of contiguous micro-batches in a virtual pipeline stage'
            f'should range in [PP={pipeline_parallel_size} , M={num_microbatches}]'
        )
        raise ValueError(msg)

    '''
    负责消除尾部气泡、确定张量形状、以及计算预热/稳态微批次数量:
    
    
    '''
    # If the final micro-batch group has fewer micro-batches than pipeline-parallel size,
    # the pipeline will have dependency bubbles.
    '''
    尾部微批次组校验:
    
    问题背景：为什么尾部会产生气泡？
    
    假设 PP=4, G=3（每组3个micro-batch）, M=10个micro-batch:

    分组结果:
      Group 0: [F0][F1][F2]   ← 3个，填满PP
      Group 1: [F3][F4][F5]   ← 3个，填满PP
      Group 2: [F6][F7]       ← 2个！尾部组

    时间线（Group 2）:
    GPU 0: [F6][F7]      ← 2个micro-batch
    GPU 1:      [F6][F7]
    GPU 2:           [F6]   ← 只有1个！GPU 2处理完F6后，F7还没从GPU 1传来
    GPU 3:                [F6]  ← 更严重的空闲！

    问题：最后一组只有2个，但PP=4需要4个才能填满流水线
          → GPU 2,3 在尾部产生大量气泡（等待时间）
          
    final_microbatch_group_size == 0	M 被 G 整除，无尾部组	✅
    final_microbatch_group_size >= PP	尾部组足够大，能填满PP	✅
    0 < final < PP	尾部组太小，产生气泡	❌ 
    
    正确配置示例：
    PP=4, M=12:
      G=3: 12%3=0 ✅
      G=4: 12%4=0 ✅
      G=5: 12%5=2 ❌ (2 < 4)
  
    PP=4, M=14:
      G=4: 14%4=2 ❌ (2 < 4)
      G=5: 14%5=4 ✅ (4 >= 4)
      G=6: 14%6=2 ❌ (2 < 4)
    '''
    final_microbatch_group_size = num_microbatches % config.microbatch_group_size_per_vp_stage
    if 0 < final_microbatch_group_size < pipeline_parallel_size:
        msg = 'The remainder of M (the total micro-batches) divided by N (number of '
        msg += 'contiguous micro-batches in a virtual pipeline stage) should be 0, '
        msg += 'or larger than or equal to the pipeline-parallel size, but it is '
        msg += f'{final_microbatch_group_size}. '
        msg += 'Otherwise, it introduces dependency bubbles in the pipeline '
        msg += 'and reduces throughput.'
        raise RuntimeError(msg)

    '''
    模型类型识别
    用途：不同模型（GPT、T5、BERT）有不同的：
      - 前向/反向传播逻辑
      - 损失计算方式
      - 通信张量形状
    '''
    model_type = get_model_type(model[0])

    tensor_shape = [seq_length, micro_batch_size, config.hidden_size]
    '''
    上下文并行（CP）切分
    
    CP=2 时:
      原始序列长度 S = 4096
      切分后每GPU处理: 4096 // 2 = 2048
  
      为什么切分第0维（seq_length）？
      → CP 是沿序列维度切分，每个GPU处理部分token的注意力计算
      
    '''
    tensor_shape[0] = tensor_shape[0] // cp_group.size()
    '''
    序列并行（SP）切分:
    
    SP 在 TP 基础上进一步切分序列维度:

    TP=4, SP=True, CP=2:
      原始: [4096, B, H]
      CP后: [2048, B, H]
      SP后: [2048 // 4, B, H] = [512, B, H]
  
      最终每个GPU处理的张量形状: [512, B, H]
    '''
    if config.sequence_parallel:
        tensor_shape[0] = tensor_shape[0] // tp_group.size()

    # Compute number of warmup and remaining microbatches.
    # seems only used for vpp
    '''
    预热/稳态微批次计算:
    
    为什么需要"预热"（Warmup）？
    
    传统 PP 的流水线填充过程（PP=4, M=8）:

    时间 →
    t0: [F0]        ← GPU 0 开始
    t1: [F0][F1]    ← GPU 1 开始
    t2: [F0][F1][F2]  ← GPU 2 开始
    t3: [F0][F1][F2][F3]  ← GPU 3 开始，流水线填满（预热结束）
    t4: [F1][F2][F3][F4]  ← 稳态：每个时钟都有4个GPU在忙
    ...

    预热期：t0-t3，流水线逐渐填满，有气泡
    稳态期：t4+，流水线全满，无气泡
    
    VPP 的预热更复杂:
    
        VPP=2, PP=4, M=8:

        预热期需要填满：
          - 物理PP阶段：4个GPU
          - 每个GPU的虚拟阶段：2个chunk
  
        预热微批次数 > 传统PP，因为VPP交错调度需要更复杂的填充
        
    输入: M=12, PP=4, VPP=2, G=3, CP=2, TP=4, SP=True, S=4096, B=2, H=4096

┌─────────────────────────────────────────────────────────────┐
│ 1. 尾部校验                                                  │
│    12 % 3 = 0 ✅ 无尾部气泡                                  │
├─────────────────────────────────────────────────────────────┤
│ 2. 张量形状计算                                              │
│    [4096, 2, 4096]                                          │
│    CP后: [2048, 2, 4096]     (4096//2)                      │
│    SP后: [512, 2, 4096]      (2048//4)                      │
│    → 最终通信张量形状: [512, 2, 4096]                        │
├─────────────────────────────────────────────────────────────┤
│ 3. 预热/稳态计算                                              │
│    num_model_chunks = 2                                      │
│    → get_pp_rank_microbatches()                              │
│    → 计算当前rank的预热和稳态微批次数                         │
│    → 返回4个值用于后续调度循环                                │
└─────────────────────────────────────────────────────────────┘
    '''
    num_model_chunks = len(model)
    (
        total_num_microbatches,# 总微批次（考虑VPP扩展）,考虑VPP后的有效微批次数
        are_all_microbatches_in_warmup,# 是否全部在预热期,如果M太小，可能全部在预热期，无稳态
        num_warmup_microbatches,# 预热期微批次数,需要多少微批次才能填满流水线
        num_microbatches_remaining,# 稳态期微批次数,填满后剩余的微批次数（稳态计算）
    ) = get_pp_rank_microbatches(
        num_microbatches,# 原始微批次数
        num_model_chunks,# VPP虚拟阶段数 (=len(model))
        config.microbatch_group_size_per_vp_stage,# 每组大小
        forward_only=forward_only,# 是否仅前向（推理无预热）
        overlap_moe_expert_parallel_comm=config.overlap_moe_expert_parallel_comm,# MoE专家并行通信重叠
        p2p_communicator=p2p_communicator,# P2P通信器（获取PP拓扑）
    )

    '''
    这段代码是 Megatron-LM VPP（虚拟流水线并行）调度器的核心配置阶段，
    涉及选择性激活检查点、参数同步和调度表生成。我来逐层拆解：
    '''
    # Checkpoint the activations of partial Transformer layers in a number of micro-batches
    # within the maximum outstanding micro-batch backpropagations.
    # Micro-batches with the ids less than 'num_microbatches_with_partial_activation_checkpoints'
    # checkpoint partial Transformer layers (or skip checkpointing) and
    # the rest of micro-batches within a window of micro-batches checkpoint
    # all Transformer layers. The window of micro-batches is set by the maximum
    # outstanding backpropagations and becomes smaller at later pipeline stages.
    # Please refer the appendix C in https://arxiv.org/pdf/2205.05198.pdf
    '''
    选择性激活检查点（Selective Activation Checkpointing）
    
    背景：激活检查点的内存-计算权衡:
    
    标准训练（无检查点）：
       前向：保存所有中间激活值 → 反向直接使用 → 内存大，计算小
  
    激活检查点（Checkpointing）：
       前向：只保存输入，丢弃中间激活 → 反向时重新计算 → 内存小，计算大
  
    问题：VPP 交错调度下，多个 micro-batch 同时在流水线中，激活值累积 → OOM
    
    论文参考：Appendix C of 2205.05198:https://arxiv.org/pdf/2205.05198.pdf
    '''
    max_outstanding_backprops = None
    if config.num_microbatches_with_partial_activation_checkpoints is not None:
        
        '''
        + 1	稳态期额外的一个 micro-batch:
        
        为什么是 warmup + 1？
        
          流水线稳态时，同时存在的 micro-batch 数 = PP 阶段数

          但这些 micro-batch 处于不同状态：
            - 一些在前向传播
            - 一些在反向传播
  
          "最大未完成反向传播数" = 同时需要保存激活值的 micro-batch 数

          = warmup 期填满的 micro-batch + 稳态新增 1 个
        '''
        max_outstanding_backprops = num_warmup_microbatches + 1

    # Synchronize params for first two model chunks
    if config.param_sync_func is not None:
        '''
        参数同步（ZeRO-3 场景）:
        
        
        为什么只同步前两个 chunk？
        
        VPP=2 时：
          model = [chunk0, chunk1]
  
          config.param_sync_func = [sync_0, sync_1]
  
          同步 chunk0 和 chunk1 的参数
  
        为什么不是全部？
          → 可能 VPP > 2 时会有更多，但代码这里硬编码为 2
          → 或者这是预热期的预取，后续 chunk 在需要时再同步
        '''
        config.param_sync_func[0](model[0].parameters())
        config.param_sync_func[1](model[1].parameters())

    # Create a tunable schedule lookup table.
    # The schedule lookup table uses the virtual_microbatch_id to find the corresponding
    # microbatch_id and model_chunk_id. For example, the tunable schedule table for
    # PP2 N3M5 with VP2 is constructed as below:
    # virtual_microbatch_id | 0 1 2 3 4 5 6 7 8 9
    # microbatch_id         | 0 1 2 0 1 2 3 4 3 4
    # model_chunk_id        | 0 0 0 1 1 1 0 0 1 1
    '''
    调度表生成（Schedule Table）
    
    参数含义：
     PP=2：流水线并行度（2个GPU）
     N=3：每个虚拟阶段的层数？或每组 micro-batch 数？
     M=5：总 micro-batch 数
     VPP=2：虚拟流水线并行度
     
     生成的调度表：
     
     virtual_microbatch_id | 0  1  2  3  4  5  6  7  8  9
     microbatch_id         | 0  1  2  0  1  2  3  4  3  4
     model_chunk_id        | 0  0  0  1  1  1  0  0  1  1
     
     如何解读这个表？
     
     时间线（按 virtual_microbatch_id 顺序执行）：

     v_id=0: microbatch=0, chunk=0  → 处理 micro-batch 0 的虚拟阶段 0
     v_id=1: microbatch=1, chunk=0  → 处理 micro-batch 1 的虚拟阶段 0
     v_id=2: microbatch=2, chunk=0  → 处理 micro-batch 2 的虚拟阶段 0

     v_id=3: microbatch=0, chunk=1  → 处理 micro-batch 0 的虚拟阶段 1
     v_id=4: microbatch=1, chunk=1  → 处理 micro-batch 1 的虚拟阶段 1
     v_id=5: microbatch=2, chunk=1  → 处理 micro-batch 2 的虚拟阶段 1

     v_id=6: microbatch=3, chunk=0  → 处理 micro-batch 3 的虚拟阶段 0
     v_id=7: microbatch=4, chunk=0  → 处理 micro-batch 4 的虚拟阶段 0

     v_id=8: microbatch=3, chunk=1  → 处理 micro-batch 3 的虚拟阶段 1
     v_id=9: microbatch=4, chunk=1  → 处理 micro-batch 4 的虚拟阶段 1
    '''
    schedule_table = get_schedule_table(
        num_microbatches,  # M = 总 micro-batch 数
        len(model), # VPP = 虚拟阶段数
        config.microbatch_group_size_per_vp_stage # G = 每组大小
    )

    # Decouple individual lookup table for microbatch_id and model_chunk_id.
    # For example, the micro-batch table for PP2 N3M5 with VP2 is
    # virtual_microbatch_id | 0 1 2 3 4 5 6 7 8 9
    # microbatch_id         | 0 1 2 0 1 2 3 4 3 4
    # Similarly, the model chunk table is
    # virtual_microbatch_id | 0 1 2 3 4 5 6 7 8 9
    # model_chunk_id        | 0 0 0 1 1 1 0 0 1 1
    # Both tables are indexed with virtual_microbatch_id.
    '''
    解耦查找表:
    
    
    # schedule_table = [(0,0), (1,0), (2,0), (0,1), (1,1), (2,1), (3,0), (4,0), (3,1), (4,1)]
    #                    ↑mb ↑ch   ↑mb ↑ch ...

    # zip(*table) 按列解包：
    microbatch_id_table  = (0, 1, 2, 0, 1, 2, 3, 4, 3, 4)
    model_chunk_id_table = (0, 0, 0, 1, 1, 1, 0, 0, 1, 1)
    '''
    microbatch_id_table, model_chunk_id_table = zip(*schedule_table)
    
    '''
    这段代码完成了 VPP 调度前的三项关键准备：
     - 通过选择性检查点平衡内存与计算、预同步 ZeRO-3 分片参数、
     - 生成并解耦虚拟微批次调度表——将复杂的 (microbatch, chunk) 交错映射转换为简单的顺序索引查找，
     - 为后续的高效流水线执行奠定基础。
    '''

    def get_model_chunk_id(virtual_microbatch_id, forward):
        """Helper method to get the model chunk ID given the iteration number."""
        model_chunk_id = model_chunk_id_table[virtual_microbatch_id % total_num_microbatches]
        if not forward:
            model_chunk_id = num_model_chunks - model_chunk_id - 1
        return model_chunk_id

    def get_microbatch_id_in_model_chunk(iteration_id, forward):
        """Helper method to get the microbatch_id within model chunk given the iteration number."""
        assert forward
        microbatch_id_in_model_chunk = microbatch_id_table[iteration_id]
        return microbatch_id_in_model_chunk

    def num_released_microbatches(virtual_microbatch_id, model_chunk_id):
        """Helper method to count number of released (i.e. popped from input_tensors)
        microbatches for a model chunk."""
        if forward_only:  # Micro-batch is released after forward prop.
            return model_chunk_id_table[:virtual_microbatch_id].count(model_chunk_id)
        else:  # Micro-batch is released after backward prop.
            # Zero backward prop in warmup.
            if virtual_microbatch_id < num_warmup_microbatches:
                return 0
            else:
                backward_microbatch_id = virtual_microbatch_id - num_warmup_microbatches
                model_chunk_id = num_model_chunks - model_chunk_id - 1
                return model_chunk_id_table[:backward_microbatch_id].count(model_chunk_id)

    def is_first_microbatch_for_model_chunk(virtual_microbatch_id: int) -> bool:
        """Check if an iteration is the first for a model chunk."""
        if virtual_microbatch_id < total_num_microbatches:
            return microbatch_id_table[virtual_microbatch_id] == 0
        else:
            return False

    def is_last_microbatch_for_model_chunk(virtual_microbatch_id: int) -> bool:
        """Check if an iteration is the last for a model chunk."""
        if virtual_microbatch_id < total_num_microbatches:
            return microbatch_id_table[virtual_microbatch_id] == num_microbatches - 1
        else:
            return False

    def recv_tensor_from_previous_stage(virtual_microbatch_id, forward):
        """Determine if peers are sending, and where in data structure
        to put received tensors.
        Return a boolean if the pipeline stage expects to recv from peers, and the
        corresponding model_chunk_id for the received tensor.
        """
        recv = True
        # The leading pipeline stage is the first rank in fwd and the last rank in bwd.
        is_leading_pipeline_stage = (
            is_pp_first_stage(p2p_communicator.pp_group)
            if forward
            else is_pp_last_stage(p2p_communicator.pp_group)
        )

        last_model_chunk = (num_model_chunks - 1) if forward else 0

        if is_leading_pipeline_stage:
            # The leading pipeline stage is ahead of the ending pipeline stage
            # (i.e. last rank in fwd and first rank in bwd) by (pipeline_parallel_size - 1).
            # Let's consider bwd as an example with PP 4:
            #       0 1 2 3 ...
            #     0 1 2 3 ...
            #   0 1 2 3 ...
            # 0 1 2 3 ...
            if virtual_microbatch_id < (pipeline_parallel_size - 1):
                # The ending stage has not produced any tensors, so no recv will be initiated.
                recv = False
                next_model_chunk_id = get_model_chunk_id(virtual_microbatch_id + 1, forward)
            else:
                # Find the model chunk of the aligned microbatches in the ending stage.
                # For example, microbatch 0 in the ending stage is aligned with microbatch 3
                # in the leading stage.
                next_model_chunk_id = get_model_chunk_id(
                    virtual_microbatch_id - (pipeline_parallel_size - 1), forward
                )
            # Last model chunk in the final stage does not produce tensors.
            if next_model_chunk_id == last_model_chunk:
                recv = False
            if forward:
                # Model chunk id increases in forward.
                next_model_chunk_id += 1
            else:
                # Model chunk id decreases in backward.
                next_model_chunk_id -= 1
        else:
            next_model_chunk_id = get_model_chunk_id(virtual_microbatch_id + 1, forward)

        return recv, next_model_chunk_id

    def forward_step_helper_preprocess(virtual_microbatch_id, model_chunk_id, microbatch_id):
        """Preprocess for forward_step_helper"""
        # launch param synchronization for next model chunk
        # Note: Asynchronous communication tends to slow down compute.
        # To reduce idling from mismatched microbatch times, we launch
        # asynchronous communication at the same time across the
        # pipeline-parallel group.
        if config.param_sync_func is not None:
            param_sync_virtual_microbatch_id = virtual_microbatch_id + pipeline_parallel_rank
            if (
                param_sync_virtual_microbatch_id < total_num_microbatches
                and is_first_microbatch_for_model_chunk(param_sync_virtual_microbatch_id)
            ):
                param_sync_chunk_id = (
                    get_model_chunk_id(param_sync_virtual_microbatch_id, forward=True) + 1
                )
                if 1 < param_sync_chunk_id < num_model_chunks:
                    config.param_sync_func[param_sync_chunk_id](
                        model[param_sync_chunk_id].parameters()
                    )

        # forward step
        if _is_vp_first_stage(vp_stage=model_chunk_id) and is_pp_first_stage(pp_group):
            if len(input_tensors[model_chunk_id]) == len(output_tensors[model_chunk_id]):
                input_tensors[model_chunk_id].append(None)

        # For non-depth-first pipeline schedules, the first rank would buffer multiple received
        # activation tensors for a model chunk until accessed during warmup.
        # This input buffering is needed to overlap the computation with the receipt of
        # the next inputs. To index the proper buffered inputs for forword_step, we use
        # microbatch_id offset with number of released microbatches that have completed backprop.
        offset = num_released_microbatches(virtual_microbatch_id, model_chunk_id)
        input_tensor = input_tensors[model_chunk_id][microbatch_id - offset]

        return input_tensor

    def forward_step_helper_postprocess(model_chunk_id, output_tensor, num_tokens):
        """Postprocess for forward_step_helper"""
        output_tensors[model_chunk_id].append(output_tensor)

        nonlocal total_num_tokens
        total_num_tokens += num_tokens

        # If forward-only, no need to save tensors for a backward pass.
        if forward_only:
            # Release the tensor that have completed forward step.
            input_tensors[model_chunk_id].pop(0)
            output_tensors[model_chunk_id].pop()

        return

    def forward_step_helper(virtual_microbatch_id, checkpoint_activations_microbatch):
        """Helper method to run forward step with model split into chunks"""
        model_chunk_id = get_model_chunk_id(virtual_microbatch_id, forward=True)
        microbatch_id = get_microbatch_id_in_model_chunk(virtual_microbatch_id, forward=True)

        input_tensor = forward_step_helper_preprocess(
            virtual_microbatch_id, model_chunk_id, microbatch_id
        )

        output_tensor, num_tokens = forward_step(
            forward_step_func,
            data_iterator[model_chunk_id],
            model[model_chunk_id],
            num_microbatches,
            input_tensor,
            forward_data_store,
            config,
            cp_group_size=cp_size,
            collect_non_loss_data=collect_non_loss_data,
            checkpoint_activations_microbatch=checkpoint_activations_microbatch,
            is_first_microbatch=check_first_val_step(
                first_val_step,
                forward_only,
                is_first_microbatch_for_model_chunk(virtual_microbatch_id),
            ),
            current_microbatch=microbatch_id,
            vp_stage=model_chunk_id,
            is_last_stage=_is_vp_last_stage(vp_stage=model_chunk_id) and is_pp_last_stage(pp_group),
        )

        forward_step_helper_postprocess(model_chunk_id, output_tensor, num_tokens)

        return output_tensor

    def backward_step_helper_preprocess(virtual_microbatch_id, model_chunk_id):
        """Preprocess for backward_step_helper"""
        # launch grad synchronization (default)
        if config.grad_sync_func is None and is_last_microbatch_for_model_chunk(
            virtual_microbatch_id
        ):
            enable_grad_sync()
            synchronized_model_chunks.add(model_chunk_id)

        # pylint: disable=E0606
        if _is_vp_last_stage(vp_stage=model_chunk_id) and is_pp_last_stage(pp_group):
            if len(output_tensor_grads[model_chunk_id]) == 0:
                output_tensor_grads[model_chunk_id].append(None)
        input_tensor = input_tensors[model_chunk_id].pop(0)
        output_tensor = output_tensors[model_chunk_id].pop(0)
        output_tensor_grad = output_tensor_grads[model_chunk_id].pop(0)

        return input_tensor, output_tensor, output_tensor_grad

    def backward_step_helper_postprocess(virtual_microbatch_id):
        """Postprocess for backward_step_helper"""
        # launch grad synchronization (custom grad sync)
        # Note: Asynchronous communication tends to slow down compute.
        # To reduce idling from mismatched microbatch times, we launch
        # asynchronous communication at the same time across the
        # pipeline-parallel group.
        if config.grad_sync_func is not None:
            grad_sync_virtual_microbatch_id = virtual_microbatch_id - pipeline_parallel_rank
            if grad_sync_virtual_microbatch_id >= 0 and is_last_microbatch_for_model_chunk(
                grad_sync_virtual_microbatch_id
            ):
                grad_sync_chunk_id = get_model_chunk_id(
                    grad_sync_virtual_microbatch_id, forward=False
                )
                enable_grad_sync()
                config.grad_sync_func[grad_sync_chunk_id](model[grad_sync_chunk_id].parameters())
                synchronized_model_chunks.add(grad_sync_chunk_id)
        disable_grad_sync()

    def backward_step_helper(virtual_microbatch_id):
        """Helper method to run backward step with model split into chunks"""
        nonlocal output_tensor_grads
        model_chunk_id = get_model_chunk_id(virtual_microbatch_id, forward=False)

        input_tensor, output_tensor, output_tensor_grad = backward_step_helper_preprocess(
            virtual_microbatch_id, model_chunk_id
        )

        input_tensor_grad = backward_step(input_tensor, output_tensor, output_tensor_grad, config)

        backward_step_helper_postprocess(virtual_microbatch_id)

        return input_tensor_grad

    def forward_backward_helper_wrapper(
        f_virtual_microbatch_id=None,
        b_virtual_microbatch_id=None,
        pre_forward=None,
        pre_backward=None,
        post_forward=None,
        post_backward=None,
        checkpoint_activations_microbatch=None,
    ):
        """
        wrap forward_helper, backward_helper, and combined_forward_backward_helper in a unified way
        """
        if config.overlap_moe_expert_parallel_comm and not forward_only:  # Combined 1F1B path
            return combined_1f1b_schedule_for_interleaved_pipelining(
                config,
                forward_step_func,
                data_iterator,
                model,
                num_microbatches,
                forward_data_store,
                forward_step_helper_preprocess,
                forward_step_helper_postprocess,
                backward_step_helper_preprocess,
                backward_step_helper_postprocess,
                get_microbatch_id_in_model_chunk,
                get_model_chunk_id,
                partial(check_first_val_step, first_val_step, forward_only),
                is_first_microbatch_for_model_chunk,
                collect_non_loss_data,
                f_virtual_microbatch_id=f_virtual_microbatch_id,
                b_virtual_microbatch_id=b_virtual_microbatch_id,
                pre_forward=pre_forward,
                pre_backward=pre_backward,
                post_forward=post_forward,
                post_backward=post_backward,
            )
        else:  # Conventional interleaved 1F1B path
            forward_output_tensor = None
            backward_input_tensor_grad = None
            # forward pass
            if f_virtual_microbatch_id is not None:
                forward_model_chunk_id = get_model_chunk_id(f_virtual_microbatch_id, forward=True)
                if pre_forward is not None:
                    pre_forward()
                forward_output_tensor = forward_step_helper(
                    f_virtual_microbatch_id, checkpoint_activations_microbatch
                )
                if post_forward is not None:
                    forward_output_tensor = post_forward(forward_output_tensor)

            # Backward pass.
            if b_virtual_microbatch_id is not None:
                backward_model_chunk_id = get_model_chunk_id(b_virtual_microbatch_id, forward=False)
                if pre_backward is not None:
                    pre_backward()
                backward_input_tensor_grad = backward_step_helper(b_virtual_microbatch_id)
                if post_backward is not None:
                    backward_input_tensor_grad = post_backward(backward_input_tensor_grad)
            return forward_output_tensor, backward_input_tensor_grad



# main logic
- 这段代码实现了 VPP 的完整三阶段调度引擎：
  - Warmup 阶段只前向填满流水线，
  - Steady 阶段通过 1F1B 回调机制实现计算与通信重叠最大化 GPU 利用率，
  - Cooldown 阶段只反向排空流水线并同步梯度——整个设计围绕"减少气泡、重叠通信、精细内存管理"三个核心目标。
- 实现了 Warmup → Steady (1F1B) → Cooldown 三阶段流水线。


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              VPP 调度三阶段                                   │
├─────────────────────────────────────────────────────────────┤
│  1. Warmup（预热）                                            │
│     → 只执行前向传播，填满流水线                              │
│     → 无反向传播，无气泡                                      │
├─────────────────────────────────────────────────────────────┤
│  2. Steady（稳态 1F1B）                                       │
│     → 前向-反向交错执行（One Forward One Backward）          │
│     → 流水线全满，GPU利用率最高                               │
├─────────────────────────────────────────────────────────────┤
│  3. Cooldown（冷却）                                          │
│     → 只执行反向传播，排空流水线                              │
│     → 完成所有剩余微批次的梯度计算                            │
└─────────────────────────────────────────────────────────────┘

In [ ]:
    # ==============================main logic=========================================
    
    # ==============================预热阶段（Warmup）=========================================
    '''
    partial 的作用：固定 vp_size 参数，后续只需传 vp_stage
    '''
    _is_vp_first_stage = partial(
        is_vp_first_stage, vp_size=config.virtual_pipeline_model_parallel_size
    )# 判断 chunk0 是否是 VPP 首阶段
    _is_vp_last_stage = partial(
        is_vp_last_stage, vp_size=config.virtual_pipeline_model_parallel_size
    )# 判断 chunk1 是否是 VPP 末阶段
    pp_group = p2p_communicator.pp_group

    # Run warmup forward passes.
    nvtx_range_push(suffix="warmup")
    '''
    1.2-接收首个输入张量
    
    - _is_vp_first_stage(vp_stage=0)	chunk0 是 VPP 首阶段
    - is_pp_first_stage(pp_group)	当前 GPU 是 PP 首阶段
    - and	只有同时满足才接收（真正的输入端）
    '''
    input_tensors[0].append(
        '''
        - 只有 GPU 0 的 chunk0 同时是 PP首 和 VPP首 → 接收外部输入
        - 其他 chunk 的输入来自上游 stage 的 P2P 发送
        '''
        p2p_communicator.recv_forward(
            tensor_shape, _is_vp_first_stage(vp_stage=0) and is_pp_first_stage(pp_group)
        )
    )

    fwd_wait_handles = None
    fwd_wait_recv_handles = None
    bwd_wait_handles = None
    bwd_wait_recv_handles = None
    
    '''1.3-通信缓冲区大小计算'''
    '''
    首/末阶段的特殊大缓冲区：
    
    原因：首阶段:需要缓存多个 micro-batch 的输入（数据加载端）
         末阶段:需要缓存多个 micro-batch 的梯度（损失计算端）
      
    计算：G - PP + 1
      G=3, PP=2: 3 - 2 + 1 = 2
      → 首/末阶段需要2个缓冲槽，中间阶段只需1个
    '''
    if is_pp_first_stage(p2p_communicator.pp_group):
        fwd_recv_buffer_size = (
            config.microbatch_group_size_per_vp_stage - pipeline_parallel_size + 1
        )
    else:
        fwd_recv_buffer_size = 1
        
    if is_pp_last_stage(p2p_communicator.pp_group):
        bwd_recv_buffer_size = (
            config.microbatch_group_size_per_vp_stage - pipeline_parallel_size + 1
        )
    else:
        bwd_recv_buffer_size = 1
    fwd_recv_buffer = [None] * fwd_recv_buffer_size
    bwd_recv_buffer = [None] * bwd_recv_buffer_size
    recv_prev_wait_handles = []
    send_next_wait_handle = None
    send_prev_wait_handle = None
    recv_next_wait_handles = []

    '''1.4-1.4 Warmup 循环核心'''
    for k in range(num_warmup_microbatches):
        '''
        get_model_chunk_id：根据虚拟微批次 ID 和方向，查调度表得当前 chunk
        '''
        cur_model_chunk_id = get_model_chunk_id(k, forward=True)

        if config.overlap_p2p_comm_warmup_flush:
            if (
                not (
                    _is_vp_first_stage(vp_stage=cur_model_chunk_id) and is_pp_first_stage(pp_group)
                )
                and k != 0
            ):
                assert recv_prev_wait_handles, (
                    f'pp rank {pipeline_parallel_rank}, iteration {k},'
                    'should have registered recv handle'
                )
                recv_prev_wait_handle = recv_prev_wait_handles.pop(0)
                recv_prev_wait_handle.wait()

        # Determine if tensor should be received from previous stage.
        recv_prev, next_forward_model_chunk_id = recv_tensor_from_previous_stage(k, forward=True)

        # No receive in last iteration when recv iteration k+1.
        if k == (total_num_microbatches - 1):
            recv_prev = False

        # Prefetch recv for iteration k+1 for non-first ranks.
        if config.overlap_p2p_comm_warmup_flush and not is_pp_first_stage(
            p2p_communicator.pp_group
        ):
            fwd_recv_buffer[k % fwd_recv_buffer_size], fwd_wait_recv_handles = (
                p2p_communicator.send_forward_recv_forward(
                    output_tensor=None,  # No output_tensor to send.
                    recv_prev=recv_prev,
                    tensor_shape=tensor_shape,
                    overlap_p2p_comm=True,
                )
            )

            if fwd_wait_recv_handles:
                recv_prev_wait_handles.append(fwd_wait_recv_handles.pop("recv_prev"))

        '''
        选择性检查点:
        
        k=0,1,2（< threshold）: 部分检查点（省内存，多计算）
        k=3,4,5（>= threshold）: 完整检查点（多内存，省计算）
        '''
        # Decide to checkpoint all layers' activations of the current micro-batch.
        if max_outstanding_backprops is not None:
            checkpoint_activations_microbatch = (
                k % max_outstanding_backprops
                >= config.num_microbatches_with_partial_activation_checkpoints
            )
        else:
            checkpoint_activations_microbatch = None

        '''
        前向传播
        '''
        output_tensor, _ = forward_backward_helper_wrapper(
            f_virtual_microbatch_id=k,
            checkpoint_activations_microbatch=checkpoint_activations_microbatch,
        )

        '''
        末阶段截断:
        
        PP 末阶段无下游：不需要发送输出给下一个 stage
        '''
        # Don't send tensor downstream if on last stage.
        if _is_vp_last_stage(vp_stage=cur_model_chunk_id) and is_pp_last_stage(pp_group):
            output_tensor = None

        # Send and receive tensors as appropriate (send tensors computed
        # in this iteration; receive tensors for next iteration).
        '''
        非重叠通信模式
        '''
        if not config.overlap_p2p_comm_warmup_flush:
            if (
                k == (num_warmup_microbatches - 1)
                and not config.overlap_p2p_comm
                and not forward_only
                and not are_all_microbatches_in_warmup
            ):
                input_tensor_grad = None
                recv_next = True
                if is_pp_last_stage(p2p_communicator.pp_group):
                    recv_next = False
                (input_tensor, output_tensor_grad) = (
                    p2p_communicator.send_forward_backward_recv_forward_backward(
                        output_tensor,
                        input_tensor_grad,
                        recv_prev=recv_prev,
                        recv_next=recv_next,
                        tensor_shape=tensor_shape,
                    )
                )
                output_tensor_grads[num_model_chunks - 1].append(output_tensor_grad)
            else:
                input_tensor = p2p_communicator.send_forward_recv_forward(
                    output_tensor, recv_prev=recv_prev, tensor_shape=tensor_shape
                )
            if recv_prev:
                input_tensors[next_forward_model_chunk_id].append(input_tensor)
            deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)
        else:
            '''
            重叠通信模式（overlap_p2p_comm）
            
            重叠通信的核心：计算和通信并行，隐藏通信延迟
            '''
            if not is_pp_first_stage(p2p_communicator.pp_group):
                # Send only since recv prefetched.
                _, fwd_wait_handles = p2p_communicator.send_forward_recv_forward(
                    output_tensor, recv_prev=False, tensor_shape=tensor_shape, overlap_p2p_comm=True
                )
            else:  # No prefetch for first rank, so both send and recv initiated.
                '''异步非阻塞式 P2P'''
                fwd_recv_buffer[k % fwd_recv_buffer_size], fwd_wait_handles = (
                    p2p_communicator.send_forward_recv_forward(
                        output_tensor,
                        recv_prev=recv_prev,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )
            '''等待发送完成'''
            if send_next_wait_handle is not None:
                send_next_wait_handle.wait()
            if fwd_wait_handles is not None:
                send_next_wait_handle = (
                    fwd_wait_handles.pop("send_next") if "send_next" in fwd_wait_handles else None
                )
                if "recv_prev" in fwd_wait_handles:
                    recv_prev_wait_handles.append(fwd_wait_handles.pop("recv_prev"))

            deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)
            if recv_prev:
                input_tensors[next_forward_model_chunk_id].append(
                    fwd_recv_buffer[k % fwd_recv_buffer_size]
                )
                fwd_recv_buffer[(k + 1) % fwd_recv_buffer_size] = None

        if config.overlap_p2p_comm:
            if (
                k == (num_warmup_microbatches - 1)
                and not forward_only
                and not are_all_microbatches_in_warmup
            ):
                input_tensor_grad = None
                recv_next = True
                if is_pp_last_stage(p2p_communicator.pp_group):
                    recv_next = False

                (bwd_recv_buffer[-1], bwd_wait_handles) = (
                    p2p_communicator.send_backward_recv_backward(
                        input_tensor_grad,
                        recv_next=recv_next,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )
                if send_prev_wait_handle is not None:
                    send_prev_wait_handle.wait()
                if bwd_wait_handles is not None:
                    send_prev_wait_handle = (
                        bwd_wait_handles.pop("send_prev")
                        if "send_prev" in bwd_wait_handles
                        else None
                    )
                    if "recv_next" in bwd_wait_handles:
                        recv_next_wait_handles.append(bwd_wait_handles.pop("recv_next"))

                if recv_next:
                    output_tensor_grads[num_model_chunks - 1].append(bwd_recv_buffer[-1])
    nvtx_range_pop(suffix="warmup")

    # ==============================稳态阶段（Steady 1F1B）=========================================
    # Run 1F1B in steady state.
    '''1F1B 模式：每个迭代同时执行一个前向 + 一个反向'''
    nvtx_range_push(suffix="steady")
    
    '''
    时间线（PP=2, VPP=2, M=5）:

    Warmup:  [F0_c0][F1_c0][F2_c0]  ← 只前向，填满流水线
    Steady:  [F3_c0][B0_c0]  ← 前向+反向交错
             [F4_c0][B1_c0]
             [F3_c1][B2_c0]
             [F4_c1][B0_c1]
             ...
    '''
    for k in range(num_microbatches_remaining):
        # Forward pass.
        forward_k = k + num_warmup_microbatches # 前向索引

        # Decide to checkpoint all layers' activations of the current micro-batch.
        if max_outstanding_backprops is not None:
            checkpoint_activations_microbatch = (
                forward_k % max_outstanding_backprops
                >= config.num_microbatches_with_partial_activation_checkpoints
            )
        else:
            checkpoint_activations_microbatch = None

        cur_model_chunk_id = get_model_chunk_id(forward_k, forward=True)
        if config.overlap_p2p_comm:

            backward_k = k # 反向索引

            # Sync forward recv
            # =======重叠通信的回调函数========
            '''
            pp_pre_forward：前向预处理
            '''
            def pp_pre_forward(vp_stage=None):
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(forward_k, forward=True)
                
                '''1. 等待接收前向输入（同步）'''
                if not (_is_vp_first_stage(vp_stage=vp_stage) and is_pp_first_stage(pp_group)):
                    if config.overlap_p2p_comm_warmup_flush:
                        assert recv_prev_wait_handles, (
                            f'pp rank {pipeline_parallel_rank}, fwd iteration {forward_k}, '
                            'should have registered recv handle'
                        )
                        recv_prev_wait_handle = recv_prev_wait_handles.pop(0)
                        recv_prev_wait_handle.wait() # 阻塞等待输入到达
                    else:
                        if recv_prev_wait_handles is not None and recv_prev_wait_handles:
                            recv_prev_wait_handle = recv_prev_wait_handles.pop(0)
                            recv_prev_wait_handle.wait()

                ''' 2. 释放上一个输出张量（省内存）'''
                deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)

            '''前向后处理'''
            # Async forward send / receive
            def pp_post_forward(output_tensor, vp_stage=None):
                nonlocal send_next_wait_handle
                nonlocal fwd_recv_buffer
                nonlocal fwd_wait_handles
                nonlocal recv_prev_wait_handles
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(forward_k, forward=True)
                '''1. 末阶段截断'''
                # Last virtual stage no activation tensor to send.
                if _is_vp_last_stage(vp_stage=vp_stage) and is_pp_last_stage(pp_group):
                    output_tensor = None

                recv_prev, next_forward_model_chunk_id = recv_tensor_from_previous_stage(
                    forward_k, forward=True
                )

                # If last iteration, don't receive; we already received one extra
                # before the start of the for loop.
                if k == (num_microbatches_remaining - 1):
                    recv_prev = False

                # Send activation tensor to the next stage and receive activation tensor from the
                # previous stage
                '''2. 发起异步 P2P（发送输出 + 预接收下一个输入）'''
                fwd_recv_buffer[forward_k % fwd_recv_buffer_size], fwd_wait_handles = (
                    p2p_communicator.send_forward_recv_forward(
                        output_tensor,
                        recv_prev=recv_prev,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )
                '''3. 等待上一次发送完成（避免积压）'''
                if send_next_wait_handle is not None:
                    send_next_wait_handle.wait()
                if fwd_wait_handles is not None:
                    send_next_wait_handle = (
                        fwd_wait_handles.pop("send_next")
                        if "send_next" in fwd_wait_handles
                        else None
                    )
                    '''4. 缓存接收句柄（供下次 pre_forward 使用）'''
                    if "recv_prev" in fwd_wait_handles:
                        recv_prev_wait_handles.append(fwd_wait_handles.pop("recv_prev"))
                # assert fwd_wait_handles is not None

                # Put input_tensor and output_tensor_grad in data structures in the
                # right location.
                if recv_prev:
                    input_tensors[next_forward_model_chunk_id].append(
                        fwd_recv_buffer[forward_k % fwd_recv_buffer_size]
                    )
                    fwd_recv_buffer[(forward_k + 1) % fwd_recv_buffer_size] = None

                return output_tensor

            '''反向预处理'''
            # Sync backward recv
            def pp_pre_backward(vp_stage=None):
                nonlocal recv_next_wait_handles
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(backward_k, forward=False)
                '''等待接收反向梯度（来自下游 stage）'''
                if not (_is_vp_last_stage(vp_stage=vp_stage) and is_pp_last_stage(pp_group)):
                    if config.overlap_p2p_comm_warmup_flush:
                        assert recv_next_wait_handles, (
                            f'pp rank {pipeline_parallel_rank}, bwd iteration {backward_k}, '
                            'should have registered recv next handle'
                        )
                        recv_next_wait_handle = recv_next_wait_handles.pop(0)
                        recv_next_wait_handle.wait()
                    else:
                        if recv_next_wait_handles is not None and recv_next_wait_handles:
                            recv_next_wait_handle = recv_next_wait_handles.pop(0)
                            recv_next_wait_handle.wait()

            '''反向后处理'''
            # Async backward send / receive
            def pp_post_backward(input_tensor_grad, vp_stage=None):
                nonlocal send_prev_wait_handle
                nonlocal bwd_wait_handles
                nonlocal recv_next_wait_handles
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(backward_k, forward=False)
                '''1. 首阶段截断（无梯度可发送）'''
                # First virtual stage no activation gradient tensor to send.
                if _is_vp_first_stage(vp_stage=vp_stage) and is_pp_first_stage(pp_group):
                    input_tensor_grad = None

                recv_next, next_backward_model_chunk_id = recv_tensor_from_previous_stage(
                    backward_k, forward=False
                )

                '''2. 发起异步 P2P（发送梯度 + 预接收下一个梯度）'''
                (bwd_recv_buffer[backward_k % bwd_recv_buffer_size], bwd_wait_handles) = (
                    p2p_communicator.send_backward_recv_backward(
                        input_tensor_grad,
                        recv_next=recv_next,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )
                if send_prev_wait_handle is not None:
                    send_prev_wait_handle.wait()
                if bwd_wait_handles is not None:
                    send_prev_wait_handle = (
                        bwd_wait_handles.pop("send_prev")
                        if "send_prev" in bwd_wait_handles
                        else None
                    )
                    if "recv_next" in bwd_wait_handles:
                        recv_next_wait_handles.append(bwd_wait_handles.pop("recv_next"))

                # Put input_tensor and output_tensor_grad in data structures in the
                # right location.

                '''3. 缓存接收的梯度'''
                if recv_next:
                    output_tensor_grads[next_backward_model_chunk_id].append(
                        bwd_recv_buffer[backward_k % bwd_recv_buffer_size]
                    )
                    bwd_recv_buffer[(backward_k + 1) % bwd_recv_buffer_size] = None
                return input_tensor_grad

            '''1F1B 核心调用'''
            '''
            回调机制：forward_backward_helper_wrapper 在适当时机调用 pre/post 钩子，实现计算与通信重叠
            '''
            output_tensor, input_tensor_grad = forward_backward_helper_wrapper(
                f_virtual_microbatch_id=forward_k,
                b_virtual_microbatch_id=backward_k,
                pre_forward=pp_pre_forward,    # 前向预处理（同步接收）
                pre_backward=pp_pre_backward,  # 反向预处理（同步接收）
                
                post_forward=pp_post_forward,  # 前向后处理（异步发送）
                post_backward=pp_post_backward,# 反向后处理（异步发送）
                checkpoint_activations_microbatch=checkpoint_activations_microbatch,
            )

        else:  # No p2p overlap.
            backward_k = k
            output_tensor, input_tensor_grad = forward_backward_helper_wrapper(
                f_virtual_microbatch_id=forward_k,
                b_virtual_microbatch_id=backward_k,
                checkpoint_activations_microbatch=checkpoint_activations_microbatch,
            )
            # Send output_tensor and input_tensor_grad, receive input_tensor
            # and output_tensor_grad.

            # Determine if current stage has anything to send in either direction,
            # otherwise set tensor to None.
            forward_model_chunk_id = get_model_chunk_id(forward_k, forward=True)
            if _is_vp_last_stage(vp_stage=forward_model_chunk_id) and is_pp_last_stage(pp_group):
                output_tensor = None

            backward_model_chunk_id = get_model_chunk_id(backward_k, forward=False)
            if _is_vp_first_stage(vp_stage=backward_model_chunk_id) and is_pp_first_stage(pp_group):
                input_tensor_grad = None

            recv_prev, next_forward_model_chunk_id = recv_tensor_from_previous_stage(
                forward_k, forward=True
            )

            recv_next, next_backward_model_chunk_id = recv_tensor_from_previous_stage(
                backward_k, forward=False
            )

            # If last iteration, don't receive; we already received one extra
            # before the start of the for loop.
            if k == (num_microbatches_remaining - 1):
                recv_prev = False

            # Communicate tensors.
            (input_tensor, output_tensor_grad) = (
                p2p_communicator.send_forward_backward_recv_forward_backward(
                    output_tensor,
                    input_tensor_grad,
                    recv_prev=recv_prev,
                    recv_next=recv_next,
                    tensor_shape=tensor_shape,
                )
            )
            deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)
            # Put input_tensor and output_tensor_grad in data structures in the
            # right location.
            if recv_prev:
                input_tensors[next_forward_model_chunk_id].append(input_tensor)
            if recv_next:
                output_tensor_grads[next_backward_model_chunk_id].append(output_tensor_grad)

    deallocate_output_tensor(output_tensor, config.deallocate_pipeline_outputs)
    nvtx_range_pop(suffix="steady")

    # ==============================冷却阶段（Cooldown）=========================================
    # Run cooldown backward passes (flush out pipeline) for the last model chunk.
    nvtx_range_push(suffix="cooldown")
    curr_vp_stage = config.virtual_pipeline_model_parallel_size - 1
    if not forward_only:
        '''
        等待未完成通信:
         确保所有异步反向通信完成，避免数据竞争
        
        '''
        if bwd_wait_handles is not None:
            for bwd_wait_handle in bwd_wait_handles.values():
                bwd_wait_handle.wait()

        if are_all_microbatches_in_warmup:
            output_tensor_grads[num_model_chunks - 1].append(
                p2p_communicator.recv_backward(
                    tensor_shape,
                    is_last_stage=(
                        _is_vp_last_stage(vp_stage=curr_vp_stage) and is_pp_last_stage(pp_group)
                    ),
                )
            )
        # ========冷却循环=========
        '''
        
        
        只执行反向传播，无新的前向传播
        
        时间线:

        Steady结束: [F4_c1][B3_c1]  ← 最后一个前向已完成
        Cooldown:   [B4_c1]         ← 只反向，排空流水线
                    [B4_c0]
                    [B3_c0]
                    ...
        '''
        for k in range(num_microbatches_remaining, total_num_microbatches):
            cur_model_chunk_id = get_model_chunk_id(k, forward=False)
            if (
                not (_is_vp_last_stage(vp_stage=cur_model_chunk_id) and is_pp_last_stage(pp_group))
                and k != 0
            ):
                if config.overlap_p2p_comm_warmup_flush:
                    assert recv_next_wait_handles, (
                        f'pp rank {pipeline_parallel_rank}, backward iteration {k}, '
                        'should have registered recv next handle'
                    )
                    recv_next_wait_handle = recv_next_wait_handles.pop(0)
                    recv_next_wait_handle.wait()
                else:
                    if recv_next_wait_handles is not None and recv_next_wait_handles:
                        recv_next_wait_handle = recv_next_wait_handles.pop(0)
                        recv_next_wait_handle.wait()

            recv_next, next_backward_model_chunk_id = recv_tensor_from_previous_stage(
                k, forward=False
            )

            if k == (total_num_microbatches - 1):
                recv_next = False

            # Prefetch recv for backward iteration k+1 for non last ranks.
            if config.overlap_p2p_comm_warmup_flush and not is_pp_last_stage(
                p2p_communicator.pp_group
            ):
                bwd_recv_buffer[k % bwd_recv_buffer_size], bwd_wait_recv_handles = (
                    p2p_communicator.send_backward_recv_backward(
                        input_tensor_grad=None,  # No input_tensor_grad to send.
                        recv_next=recv_next,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )

                if bwd_wait_recv_handles:
                    recv_next_wait_handles.append(bwd_wait_recv_handles.pop("recv_next"))

            _, input_tensor_grad = forward_backward_helper_wrapper(b_virtual_microbatch_id=k)

            # First virtual stage no activation gradient tensor to send.
            if _is_vp_first_stage(vp_stage=cur_model_chunk_id) and is_pp_first_stage(pp_group):
                input_tensor_grad = None

            if config.overlap_p2p_comm_warmup_flush:
                if not is_pp_last_stage(p2p_communicator.pp_group):
                    _, bwd_wait_handles = p2p_communicator.send_backward_recv_backward(
                        input_tensor_grad,
                        recv_next=False,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                else:
                    bwd_recv_buffer[k % bwd_recv_buffer_size], bwd_wait_handles = (
                        p2p_communicator.send_backward_recv_backward(
                            input_tensor_grad,
                            recv_next=recv_next,
                            tensor_shape=tensor_shape,
                            overlap_p2p_comm=True,
                        )
                    )

                if send_prev_wait_handle is not None:
                    send_prev_wait_handle.wait()
                if bwd_wait_handles is not None:
                    send_prev_wait_handle = (
                        bwd_wait_handles.pop("send_prev")
                        if "send_prev" in bwd_wait_handles
                        else None
                    )
                    if "recv_next" in bwd_wait_handles:
                        recv_next_wait_handles.append(bwd_wait_handles.pop("recv_next"))
                if recv_next:
                    output_tensor_grads[next_backward_model_chunk_id].append(
                        bwd_recv_buffer[k % bwd_recv_buffer_size]
                    )
                    bwd_recv_buffer[(k + 1) % bwd_recv_buffer_size] = None

            else:
                output_tensor_grad = p2p_communicator.send_backward_recv_backward(
                    input_tensor_grad, recv_next=recv_next, tensor_shape=tensor_shape
                )

                if recv_next:
                    output_tensor_grads[next_backward_model_chunk_id].append(output_tensor_grad)

        if send_prev_wait_handle is not None:
            send_prev_wait_handle.wait()

        '''
        梯度同步
        
        逐个 chunk 同步梯度，确保所有虚拟阶段的梯度都完成聚合
        '''
        # Launch any remaining grad reductions.
        enable_grad_sync() # 退出 no_sync 模式，触发 All-Reduce
        if config.grad_sync_func is not None:
            for model_chunk_id in range(num_model_chunks):
                if model_chunk_id not in synchronized_model_chunks:
                    config.grad_sync_func[model_chunk_id](model[model_chunk_id].parameters())
                    synchronized_model_chunks.add(model_chunk_id)
    nvtx_range_pop(suffix="cooldown")

    # ==============================收尾阶段（Misc）=========================================
    nvtx_range_push(suffix="misc")
    
    '''
    防御性编程：确保无残留异步操作，防止内存泄漏
    '''
    assert (
        not recv_prev_wait_handles
    ), 'recv_prev_wait_handles should be cleared at the end of a step'
    assert (
        not recv_next_wait_handles
    ), 'recv_next_wait_handles should be cleared at the end of a step'

    '''梯度最终化'''
    if config.finalize_model_grads_func is not None and not forward_only:

        # If defer_embedding_wgrad_compute is enabled we need to do the
        # weight gradient GEMM's here.
        '''1. 完成嵌入层权重梯度计算'''
        finish_embedding_wgrad_compute(
            config, embedding_module, p2p_communicator.is_pp_last_stage, tp_group
        )

        # Finalize model grads (perform full grad all-reduce / reduce-scatter for
        # data parallelism, layernorm all-reduce for sequence parallelism, and
        # embedding all-reduce for pipeline parallelism).
        '''2. 全局梯度聚合（TP/CP/DP All-Reduce）'''
        config.finalize_model_grads_func(
            model,
            total_num_tokens if config.calculate_per_token_loss else None,
            pg_collection=pg_collection,
            force_all_reduce=force_all_reduce,
        )

    if getattr(config, 'fine_grained_activation_offloading', False):
        off_interface.reset()
        
    '''恢复推理模式配置'''    
    # Restore config.grad_sync_func and config.param_sync_func.
    if forward_only:
        '''
        恢复之前暂存的同步函数，供下次训练使用
        '''
        config.grad_sync_func, config.param_sync_func = grad_sync_func, param_sync_func

    '''计时结束'''
    if config.timers is not None:
        config.timers('forward-backward').stop()

    if (
        hasattr(config, 'cuda_graph_impl')
        and config.cuda_graph_impl == "local"
        and CudaGraphScope.full_iteration not in config.cuda_graph_scope
    ):
        create_cudagraphs()
    nvtx_range_pop(suffix="misc")

    return forward_data_store


## 完整时序图

In [ ]:
PP=2, VPP=2, M=5, G=3

时间 →
GPU 0 (chunk0+chunk2):
  Warmup:  [F0_c0][F1_c0][F2_c0]                    ← 只前向，填满流水线
  Steady:  [F3_c0][B0_c0][F4_c0][B1_c0][F3_c2][B2_c0][F4_c2][B0_c2][B1_c2][B2_c2]
            ↑1F1B交错，计算与通信重叠
  Cooldown: [B3_c2][B4_c2][B3_c0][B4_c0]            ← 只反向，排空流水线

GPU 1 (chunk1+chunk3):
  Warmup:  [F0_c1][F1_c1][F2_c1]
  Steady:  [F3_c1][B0_c1][F4_c1][B1_c1][F3_c3][B2_c1][F4_c3][B0_c3][B1_c3][B2_c3]
  Cooldown: [B3_c3][B4_c3][B3_c1][B4_c1]

通信:
  ←→ 表示 P2P 发送/接收，与计算重叠（overlap_p2p_comm）

## forward_backward_helper_wrapper
- 这个包装器是 VPP 调度引擎的核心抽象层：
  - 通过统一的回调接口（pre/post forward/backward）将计算步骤与通信步骤解耦，
  - 支持常规 1F1B 的细粒度计算-通信重叠，以及 MoE 场景下更粗粒度的 combined 调度——本质上是"策略模式"，让外层循环无需关心具体的重叠实现细节。
- 这段代码是 Megatron-LM 中 <font color='red'>VPP（虚拟流水线并行）的前向-反向传播调度包装器</font>，实现了两种执行路径的统一抽象：
- 常规 1F1B 路径 
- MoE 专家并行通信重叠路径

| 参数                                  | 类型              | 含义                  |
| ----------------------------------- | --------------- | ------------------- |
| `f_virtual_microbatch_id`           | `int/None`      | 前向传播的虚拟微批次 ID       |
| `b_virtual_microbatch_id`           | `int/None`      | 反向传播的虚拟微批次 ID       |
| `pre_forward`                       | `callable/None` | 前向预处理回调（如等待 P2P 接收） |
| `post_forward`                      | `callable/None` | 前向后处理回调（如发起 P2P 发送） |
| `pre_backward`                      | `callable/None` | 反向预处理回调             |
| `post_backward`                     | `callable/None` | 反向后处理回调             |
| `checkpoint_activations_microbatch` | `bool/None`     | 是否对此微批次启用激活检查点      |
- 关键设计：f_* 和 b_* 可以同时为 None、只有一个为 None、或同时有值——支持灵活的调度模式。

### 整体架构


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              forward_backward_helper_wrapper                 │
│                    （统一调度包装器）                         │
├─────────────────────────────────────────────────────────────┤
│  条件分支                                                    │
│  ├── overlap_moe_expert_parallel_comm=True                  │
│  │   → combined_1f1b_schedule_for_interleaved_pipelining   │
│  │   → MoE 专家并行 + 通信重叠优化路径                        │
│  │                                                          │
│  └── 常规路径（else）                                         │
│      → 独立执行前向 + 独立执行反向                            │
│      → 通过 pre/post 回调实现通信重叠                         │
└─────────────────────────────────────────────────────────────┘

### 路径一：MoE 专家并行 + 通信重叠（Combined 1F1B）

#### 什么是 MoE（Mixture of Experts）？

In [ ]:
标准 Transformer:
  FFN: 所有 token 经过同一个前馈网络
  
MoE:
  多个专家网络（Expert FFNs）
  Router: 决定每个 token 去哪个专家
  → 每个 token 只经过少数专家（如 Top-2）
  
问题: 专家分布在不同 GPU 上 → All-to-All 通信开销大

#### 为什么需要 Combined 1F1B？

In [ ]:
常规 1F1B:
  [F][B][F][B]... 前向和反向交替
  
MoE 问题:
  前向: token → Router → All-to-All 分发到专家 GPU → 计算 → All-to-All 聚合
  反向: 梯度 → All-to-All 分发 → 计算 → All-to-All 聚合
  
  通信密集！常规调度无法重叠 MoE 通信
  
Combined 1F1B:
  将前向和反向的 MoE 通信合并，统一调度
  → 在一个更大的窗口内重叠计算和通信

#### 调用参数
- 特点：将大量上下文传入，由 combined_1f1b_schedule 内部统一调度前向和反向的 MoE 通信

In [ ]:
return combined_1f1b_schedule_for_interleaved_pipelining(
    config,
    forward_step_func,           # 前向步函数
    data_iterator,               # 数据迭代器（VPP 列表）
    model,                       # 模型 chunk 列表
    num_microbatches,            # 总微批次数
    forward_data_store,          # 前向数据存储
    forward_step_helper_preprocess,   # 前向预处理
    forward_step_helper_postprocess,  # 前向后处理
    backward_step_helper_preprocess,  # 反向预处理
    backward_step_helper_postprocess, # 反向后处理
    get_microbatch_id_in_model_chunk, # 查表函数
    get_model_chunk_id,          # 查 chunk ID
    partial(check_first_val_step, first_val_step, forward_only),  # 首验证步检查
    is_first_microbatch_for_model_chunk,  # 是否 chunk 的首微批次
    collect_non_loss_data,       # 收集非损失数据
    f_virtual_microbatch_id=f_virtual_microbatch_id,
    b_virtual_microbatch_id=b_virtual_microbatch_id,
    pre_forward=pre_forward,
    pre_backward=pre_backward,
    post_forward=post_forward,
    post_backward=post_backward,
)

### 路径二：常规交错 1F1B（Conventional Interleaved）
#### 有重叠模式（异步）

In [ ]:
# Steady 阶段传入的回调
pre_forward=pp_pre_forward      # 等待接收输入
post_forward=pp_post_forward  # 发送输出 + 预接收下一个
pre_backward=pp_pre_backward  # 等待接收梯度
post_backward=pp_post_backward # 发送梯度 + 预接收下一个

# 执行时序：
pre_forward()           # 等待上次预接收的输入到达（同步）
forward_step_helper()   # 计算（耗时）
post_forward()          # 发送输出 + 发起下次预接收（异步返回）
                        # ← 计算和通信重叠！

#### 无重叠模式（同步）

In [ ]:
# pre/post 都为 None
forward_output_tensor = forward_step_helper(v_id, checkpoint)
# 同步阻塞，计算完再通信

In [ ]:
时间 →
无重叠:
  [等待输入]→[计算F]→[发送输出]→[等待输入]→[计算F]→[发送输出]
  ↑同步阻塞  ↑同步阻塞  ↑同步阻塞

有重叠:
  [等待输入]→[计算F]→[发送输出+预接收]→[计算F]→[发送输出+预接收]
            ↑计算期间，通信在后台进行

#### 调度模式组合

| `f_id` | `b_id` | 场景    | 说明             |
| ------ | ------ | ----- | -------------- |
| `None` | `None` | 无操作   | 占位/跳过          |
| `k`    | `None` | 仅前向   | Warmup 阶段      |
| `None` | `k`    | 仅反向   | Cooldown 阶段    |
| `fw_k` | `bw_k` | 前向+反向 | Steady 1F1B 阶段 |


#### Warmup 调用示例

In [ ]:
# 只传前向 ID
forward_backward_helper_wrapper(
    f_virtual_microbatch_id=k,   # 如 k=0,1,2
    b_virtual_microbatch_id=None,
    
    pre_forward=pp_pre_forward,
    post_forward=pp_post_forward,
)
# 只执行前向，返回 (output_tensor, None)

#### Steady 1F1B 调用示例

In [ ]:
# 同时传前向和反向 ID
forward_backward_helper_wrapper(
    f_virtual_microbatch_id=forward_k,   # 如 3
    b_virtual_microbatch_id=backward_k,  # 如 0
    
    pre_forward=pp_pre_forward,
    post_forward=pp_post_forward,
     '''增加了以下内容'''
    pre_backward=pp_pre_backward,
    post_backward=pp_post_backward,
)
# 先执行前向，再执行反向，返回 (output_tensor, input_grad)

In [ ]:
   def forward_backward_helper_wrapper(
        f_virtual_microbatch_id=None,
        b_virtual_microbatch_id=None,
        pre_forward=None,
        pre_backward=None,
       
        post_forward=None,
        post_backward=None,
        checkpoint_activations_microbatch=None,
    ):
        '''路径一：MoE 专家并行 + 通信重叠（Combined 1F1B）'''
        """
        wrap forward_helper, backward_helper, and combined_forward_backward_helper in a unified way
        """
        
        '''
        overlap_moe_expert_parallel_comm=True	启用 MoE 专家并行通信重叠
        not forward_only	训练模式（推理不需要反向传播，无需重叠）
        '''
        if config.overlap_moe_expert_parallel_comm and not forward_only:  # Combined 1F1B path
            return combined_1f1b_schedule_for_interleaved_pipelining(
                config,
                forward_step_func,
                data_iterator,
                model,
                num_microbatches,
                forward_data_store,
                forward_step_helper_preprocess,
                forward_step_helper_postprocess,
                backward_step_helper_preprocess,
                backward_step_helper_postprocess,
                get_microbatch_id_in_model_chunk,
                get_model_chunk_id,
                partial(check_first_val_step, first_val_step, forward_only),
                is_first_microbatch_for_model_chunk,
                collect_non_loss_data,
                f_virtual_microbatch_id=f_virtual_microbatch_id,
                b_virtual_microbatch_id=b_virtual_microbatch_id,
                pre_forward=pre_forward,
                pre_backward=pre_backward,
                post_forward=post_forward,
                post_backward=post_backward,
            )
        else:  # Conventional interleaved 1F1B path
            '''路径二：常规交错 1F1B（Conventional Interleaved）'''
            forward_output_tensor = None
            backward_input_tensor_grad = None
            # ===前向传播===
            # forward pass 
            if f_virtual_microbatch_id is not None:
                '''1. 确定当前 model chunk'''
                forward_model_chunk_id = get_model_chunk_id(f_virtual_microbatch_id, forward=True)
                
                '''2. 预处理（如等待 P2P 接收完成）'''
                if pre_forward is not None:
                    pre_forward()
                    
                '''3. 执行前向计算'''
                forward_output_tensor = forward_step_helper(
                    f_virtual_microbatch_id, checkpoint_activations_microbatch
                )
                '''4. 后处理（如发起 P2P 发送）'''
                if post_forward is not None:
                    forward_output_tensor = post_forward(forward_output_tensor)

                    
            # ===反向传播==
            # Backward pass.
            if b_virtual_microbatch_id is not None:
                '''1. 确定当前 model chunk'''
                backward_model_chunk_id = get_model_chunk_id(b_virtual_microbatch_id, forward=False)
                
                '''2. 预处理（如等待梯度接收）'''
                if pre_backward is not None:
                    pre_backward()
                    
                '''3. 执行反向计算'''
                backward_input_tensor_grad = backward_step_helper(b_virtual_microbatch_id)
                
                '''4. 后处理（如发送梯度）'''
                if post_backward is not None:
                    backward_input_tensor_grad = post_backward(backward_input_tensor_grad)
            return forward_output_tensor, backward_input_tensor_grad



### 举例解释
- 这个包装器通过"条件分支 + 回调钩子"实现四种调度模式的统一：
  - Warmup只前向
  - Steady前后向一起
  - Cooldown只反向
  - MoE特殊合并——外层循环只需传不同的ID和回调，无需关心内部是同步还是异步、是常规还是MoE。

场景设定：

- PP=2, VPP=2, M=4（4个micro-batch）
- model = [chunk0, chunk1]  ← 一个GPU上两个不连续虚拟阶段

#### 例子1：Warmup 阶段（只前向）

In [ ]:
# 外层循环调用
for k in range(3):  # num_warmup_microbatches=3
    output_tensor, _ = forward_backward_helper_wrapper(
        f_virtual_microbatch_id=k,   # k=0,1,2
        b_virtual_microbatch_id=None,  # 不执行反向
        pre_forward=pp_pre_forward,
        post_forward=pp_post_forward,
    )

##### k=0 时的执行流程

In [ ]:
f_virtual_microbatch_id = 0
b_virtual_microbatch_id = None  # 不执行反向

# 进入函数
forward_output_tensor = None
backward_input_tensor_grad = None

# ========== 前向分支 ==========
if f_virtual_microbatch_id is not None:  # 0 is not None → 执行
    forward_model_chunk_id = get_model_chunk_id(0, forward=True)
    # 查表得: virtual_id=0 → model_chunk_id=0
    
    # Step 1: 预处理（等待输入）
    pre_forward()  # pp_pre_forward()
    # 内部: 等待 recv_prev_wait_handles[0] 完成
    # 从上游stage接收输入张量
    
    # Step 2: 前向计算
    forward_output_tensor = forward_step_helper(0, checkpoint_activations_microbatch)
    # chunk0 处理 micro-batch 0 的数据
    # 输出 shape: [seq_len, micro_batch_size, hidden_size]
    
    # Step 3: 后处理（发送输出）
    forward_output_tensor = post_forward(forward_output_tensor)
    # pp_post_forward()
    # 内部: 发送输出给下游stage + 预接收下一个输入

# ========== 反向分支 ==========
if b_virtual_microbatch_id is not None:  # None → 跳过

# 返回
return forward_output_tensor, None
# 即: (tensor([...]), None)

#### 例子2：Steady 1F1B 阶段（前向+反向）

In [ ]:
# 外层循环调用
for k in range(1):  # num_microbatches_remaining=1
    forward_k = k + 3  # 3
    backward_k = k     # 0
    
    output_tensor, input_grad = forward_backward_helper_wrapper(
        f_virtual_microbatch_id=3,
        b_virtual_microbatch_id=0,
        pre_forward=pp_pre_forward,
        post_forward=pp_post_forward,
        pre_backward=pp_pre_backward,
        post_backward=pp_post_backward,
    )

##### 执行流程

In [ ]:
f_virtual_microbatch_id = 3
b_virtual_microbatch_id = 0

forward_output_tensor = None
backward_input_tensor_grad = None

# ========== 前向分支 ==========
if 3 is not None:
    forward_model_chunk_id = get_model_chunk_id(3, forward=True)
    # 查表: virtual_id=3 → microbatch_id=0, model_chunk_id=1
    
    pre_forward()  # 等待 chunk1 的输入到达
    
    # chunk1 处理 micro-batch 0
    forward_output_tensor = forward_step_helper(3, checkpoint)
    
    # 末阶段判断
    if _is_vp_last_stage(vp_stage=1) and is_pp_last_stage(pp_group):
        forward_output_tensor = None  # 最后一stage，不发送输出
    
    post_forward(forward_output_tensor)  # 发送（如果是None就不发）

# ========== 反向分支 ==========
if 0 is not None:
    backward_model_chunk_id = get_model_chunk_id(0, forward=False)
    # 查表: virtual_id=0 → microbatch_id=0, model_chunk_id=0
    
    pre_backward()  # 等待 chunk0 的梯度到达（来自下游stage）
    
    # chunk0 反向传播 micro-batch 0
    backward_input_tensor_grad = backward_step_helper(0)
    
    # 首阶段判断
    if _is_vp_first_stage(vp_stage=0) and is_pp_first_stage(pp_group):
        backward_input_tensor_grad = None  # 第一stage，不发送梯度
    
    post_backward(backward_input_tensor_grad)  # 发送梯度给上游

# 返回
return forward_output_tensor, backward_input_tensor_grad

#### 例子3：Cooldown 阶段（只反向）

In [ ]:
# 外层循环调用
for k in range(3, 4):  # 剩余的反向
    _, input_grad = forward_backward_helper_wrapper(
        f_virtual_microbatch_id=None,   # 不执行前向
        b_virtual_microbatch_id=k,       # k=3
        pre_backward=pp_pre_backward,
        post_backward=pp_post_backward,
    )

##### 执行流程

In [ ]:
f_virtual_microbatch_id = None
b_virtual_microbatch_id = 3

# ========== 前向分支 ==========
if None is not None:  # False → 跳过

# ========== 反向分支 ==========
if 3 is not None:
    backward_model_chunk_id = get_model_chunk_id(3, forward=False)
    # 查表: virtual_id=3 → model_chunk_id=1
    
    pre_backward()  # 等待梯度
    
    backward_input_tensor_grad = backward_step_helper(3)
    
    post_backward(backward_input_tensor_grad)

# 返回
return None, backward_input_tensor_grad

#### 例子4：无重叠模式（同步通信）

In [ ]:
# 配置: overlap_p2p_comm = False
# 此时 pre/post 回调为 None

output_tensor, _ = forward_backward_helper_wrapper(
    f_virtual_microbatch_id=0,
    b_virtual_microbatch_id=None,
    pre_forward=None,      # 无预处理
    post_forward=None,     # 无后处理
)

# 执行流程:
if 0 is not None:
    forward_model_chunk_id = get_model_chunk_id(0, forward=True)
    
    # pre_forward is None → 跳过
    
    forward_output_tensor = forward_step_helper(0, checkpoint)
    # 计算完成
    
    # post_forward is None → 跳过
    
# 返回后，外层手动同步通信:
input_tensor = p2p_communicator.send_forward_recv_forward(
    output_tensor, recv_prev=True, tensor_shape=tensor_shape
)
# 阻塞等待！

##### 对比有重叠模式：

In [ ]:
# 有重叠: post_forward 内部发起异步通信
# 返回时通信已在后台进行
# 下次 pre_forward 才等待上次通信完成

# 无重叠: 外层手动同步通信
# 阻塞等待，GPU空闲

#### MoE 路径
- 关键区别：MoE 路径将前向和反向的通信合并调度，而不是独立处理。

In [ ]:
# 配置: overlap_moe_expert_parallel_comm = True, forward_only = False

output_tensor, input_grad = forward_backward_helper_wrapper(
    f_virtual_microbatch_id=3,
    b_virtual_microbatch_id=0,
    # ... 各种回调
)

# 由于条件满足，走 combined_1f1b_schedule_for_interleaved_pipelining
# 而不是常规路径

# 内部逻辑（简化）:
def combined_1f1b_schedule(...):
    # 1. 前向计算 + 记录 MoE All-to-All 通信句柄
    fwd_output, moe_fwd_handles = forward_with_moe_comm(...)
    
    # 2. 反向计算 + 复用/重叠 MoE 通信
    bwd_grad, moe_bwd_handles = backward_with_moe_comm(
        moe_fwd_handles  # 前向的通信状态传给反向复用
    )
    
    # 3. 统一调度前向和反向的通信
    overlap_moe_comm(moe_fwd_handles, moe_bwd_handles)
    
    return fwd_output, bwd_grad

In [ ]:
无重叠模式 (pre/post=None):
┌─────────┐     ┌─────────┐     ┌─────────┐
│ 计算F   │────→│ 同步通信 │────→│ 计算F   │
│         │阻塞 │         │阻塞 │         │
└─────────┘     └─────────┘     └─────────┘
  GPU忙          GPU空闲          GPU忙

有重叠模式 (pre/post有值):
┌─────────┐     ┌─────────┐     ┌─────────┐
│pre_wait │────→│ 计算F   │────→│post_send│
│(等上次) │     │         │     │+预接收  │
└─────────┘     └─────────┘     └─────────┘
  短暂等待        GPU忙           异步，GPU立即继续

下次迭代:
┌─────────┐
│pre_wait │← 等待上次post发起的预接收完成
│(已就绪) │   （通信在计算期间已完成！）
└─────────┘